In [ ]:
# ============================================================
# PRELIMINARY STEP: CALCULATE INITIAL COMPLIANCE VIOLATIONS
# ============================================================
#
# This script evaluates the original process models against
# their associated compliance requirement sets using the
# ProcessTreeVerify verifier.
#
# For each scenario, the script:
# - injects the compliance requirements into the process model
# - executes the compliance verification
# - extracts the detected violations
# - stores the generated violation reports
#
# The script supports executing either:
# - one selected scenario
# - all available scenarios
#
# Output:
# - compliance violation reports (.json)
#
# ============================================================

import json
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

# -----------------------------------
# Configuration
# -----------------------------------

RUN_ALL_SCENARIOS = True
SELECTED_SCENARIO = "01_awad_delivery_of_goods"

SCENARIOS = [
    "01_awad_delivery_of_goods",
    "02_de_masellis_loan_approval",
    "03_elgammal_loan_approval",
    "04_ghose_package_screening",
    "05_loebbecke_blood_collection",
    "06_BPMQ",
    "07_CRL",
    "08_DCR",
    "09_HaarmannetAL2021",
    "11_PCL",
    "12_PPSL",
    "13_Status",
    "14_SunetAl24",
    "15_WinteretAlER",
    "16_Zasadaetal23"
]

VERIFY_ALL = True

selected_requirement_ids = [
    "R1"
]

# -----------------------------------
# Dynamically locate project root
# -----------------------------------

BASE_DIR = Path.cwd().parents[1]

# -----------------------------------
# Paths
# -----------------------------------

PTV_SCRIPT = (
    BASE_DIR
    / "src"
    / "ProcessTreeVerify"
    / "python_code"
    / "test_script.py"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_before_changes"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------
# Namespaces
# -----------------------------------

NS_PROPERTIES = (
    "http://cpee.org/ns/properties/2.0"
)

NS_SUBSCRIPTIONS = (
    "http://riddl.org/ns/common-patterns/"
    "notifications-producer/2.0"
)

namespace = {
    "ns1": NS_PROPERTIES
}

# -----------------------------------
# Ensure <requirements> node
# -----------------------------------

def ensure_requirements_node(root):
    attributes_node = root.find(
        ".//ns1:attributes",
        namespace
    )

    if attributes_node is None:
        raise ValueError(
            "No <attributes> node found."
        )

    req_node = root.find(
        ".//ns1:requirements",
        namespace
    )

    if req_node is None:
        print(
            "\n[INFO] Creating "
            "<requirements> node."
        )

        req_node = ET.SubElement(
            attributes_node,
            f"{{{NS_PROPERTIES}}}requirements"
        )

    else:
        print(
            "\n[INFO] Existing "
            "<requirements> node found."
        )

    return req_node

# -----------------------------------
# Ensure <subscriptions> block
# -----------------------------------

def ensure_subscriptions(root):
    subscriptions = root.find(
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    if subscriptions is not None:
        print(
            "\n[INFO] Subscriptions block already exists."
        )
        return

    print(
        "\n[INFO] Creating subscriptions block."
    )

    subscriptions = ET.SubElement(
        root,
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    subscription = ET.SubElement(
        subscriptions,
        f"{{{NS_SUBSCRIPTIONS}}}subscription",
        {
            "id": "_compliance",
            "url": (
                "https://power.bpm.cit.tum.de/"
                "compliance/Subscriber"
            )
        }
    )

    topic = ET.SubElement(
        subscription,
        f"{{{NS_SUBSCRIPTIONS}}}topic",
        {
            "id": "description"
        }
    )

    event = ET.SubElement(
        topic,
        f"{{{NS_SUBSCRIPTIONS}}}event"
    )

    event.text = "change"

# -----------------------------------
# Run one scenario
# -----------------------------------

def run_scenario(scenario_name):
    print("\n===================================")
    print(f"RUNNING SCENARIO: {scenario_name}")
    print("===================================\n")

    xml_file = (
        BASE_DIR
        / "data"
        / "input"
        / "process_models"
        / "cpee_trees"
        / f"{scenario_name}.xml"
    )

    requirements_file = (
        BASE_DIR
        / "data"
        / "input"
        / "compliance_requirements"
        / "ast"
        / f"{scenario_name}.json"
    )

    with open(
        requirements_file,
        "r",
        encoding="utf-8"
    ) as f:
        all_requirements = json.load(f)

    if VERIFY_ALL:
        requirements = all_requirements

        print(
            "\nMode: VERIFY ALL REQUIREMENTS"
        )

    else:
        print(
            "\nMode: VERIFY SELECTED REQUIREMENTS"
        )

        missing_requirements = [
            rid
            for rid in selected_requirement_ids
            if rid not in all_requirements
        ]

        if missing_requirements:
            raise ValueError(
                f"Requirements not found: "
                f"{missing_requirements}"
            )

        requirements = {
            rid: all_requirements[rid]
            for rid in selected_requirement_ids
        }

    requirements_text = json.dumps(
        requirements,
        ensure_ascii=False
    )

    print("\nInjected requirements:")
    print(requirements_text)

    tree = ET.parse(xml_file)
    root = tree.getroot()

    req_node = ensure_requirements_node(root)
    ensure_subscriptions(root)

    req_node.text = requirements_text

    print(
        "\n[INFO] Requirements successfully injected."
    )

    with tempfile.NamedTemporaryFile(
        suffix=".xml",
        delete=False
    ) as tmp:
        temp_xml_path = Path(tmp.name)

    tree.write(
        temp_xml_path,
        encoding="utf-8",
        xml_declaration=True
    )

    print(
        f"\nTemporary XML created:\n"
        f"{temp_xml_path}"
    )

    result = subprocess.run(
        [
            "python3",
            str(PTV_SCRIPT),
            str(temp_xml_path)
        ],
        capture_output=True,
        text=True
    )

    print("\n=== STDOUT ===")
    print(result.stdout)

    print("\n=== STDERR ===")
    print(result.stderr)

    marker = "===== VIOLATION REPORT ====="

    if marker in result.stdout:
        violation_json = (
            result.stdout
            .split(marker)[-1]
            .strip()
        )

        violation_report = json.loads(
            violation_json
        )

        if VERIFY_ALL:
            selected_name = "ALL"

        else:
            selected_name = "_".join(
                selected_requirement_ids
            )

        output_file = (
            OUTPUT_DIR
            / (
                f"{scenario_name}_"
                f"{selected_name}_violations.json"
            )
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                violation_report,
                f,
                indent=4,
                ensure_ascii=False
            )

        print(
            f"\nViolation report saved to:\n"
            f"{output_file}"
        )

    else:
        print("\nNo violation report found.")

    temp_xml_path.unlink(
        missing_ok=True
    )

    print("\nTemporary XML deleted.")

# -----------------------------------
# Main execution
# -----------------------------------

if RUN_ALL_SCENARIOS:
    for scenario in SCENARIOS:
        run_scenario(scenario)

else:
    run_scenario(SELECTED_SCENARIO)

In [ ]:
# ============================================================
# ANALYSIS STEP:
# CALCULATE GLOBAL COMPLIANCE VIOLATION STATISTICS
# ============================================================
#
# This script analyzes multiple compliance violation reports
# generated from process verification runs and computes
# aggregated statistics about violated compliance patterns.
#
# For each violation report, the script:
# - extracts violated compliance requirements
# - identifies the compliance patterns involved
# - computes global and per-file statistics
# - aggregates requirement-level information
#
# The script produces:
# - global violation statistics
# - per-scenario/file pattern distributions
# - requirement-level summaries
#
# Output:
# - aggregated violation statistics (.json)
#
# ============================================================

import json
import re
from pathlib import Path
from collections import Counter, defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# CONFIGURATION
# ============================================================

# Folder containing multiple violation JSON files

INPUT_FOLDER = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_before_changes"
)

# Output summary file

OUTPUT_JSON = (
    BASE_DIR
    / "data"
    / "output"
    / "global_violation_statistics.json"
)

# Allowed compliance patterns

ALLOWED_PATTERNS = [
    "exists",
    "absence",
    "loop",
    "directly_follows",
    "leads_to",
    "precedence",
    "leads_to_absence",
    "precedence_absence",
    "parallel",
    "executed_by",
    "timed_alternative",
    "activity_sends",
    "activity_receives",
    "condition_directly_follows",
    "condition_eventually_follows",
    "data_leads_to_absence"
]

# ============================================================
# INITIALIZATION
# ============================================================

pattern_regex = re.compile(
    r'\b([a-zA-Z_][a-zA-Z0-9_]*)\s*\('
)

global_pattern_counter = Counter()

file_statistics = {}

requirement_statistics = defaultdict(list)

total_files = 0
total_violations = 0

# ============================================================
# ITERATE OVER ALL JSON FILES
# ============================================================

json_files = list(
    Path(INPUT_FOLDER).glob("*.json")
)

for json_file in json_files:

    total_files += 1

    print(f"Processing: {json_file.name}")

    try:

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            violations = json.load(f)

    except Exception as e:

        print(
            f"ERROR reading "
            f"{json_file.name}: {e}"
        )

        continue

    # --------------------------------------------------------
    # Skip invalid structures
    # --------------------------------------------------------

    if not isinstance(violations, list):

        print(
            f"Skipping {json_file.name}: "
            f"expected a list"
        )

        continue

    file_pattern_counter = Counter()

    file_violation_count = len(violations)

    total_violations += file_violation_count

    # --------------------------------------------------------
    # PROCESS EACH VIOLATION
    # --------------------------------------------------------

    for violation in violations:

        requirement_id = violation.get(
            "requirement_id",
            "UNKNOWN"
        )

        requirement = violation.get(
            "requirement",
            ""
        )

        found_patterns = pattern_regex.findall(
            requirement
        )

        valid_patterns = [
            p for p in found_patterns
            if p in ALLOWED_PATTERNS
        ]

        # ----------------------------------------------------
        # Global statistics
        # ----------------------------------------------------

        for pattern in valid_patterns:

            global_pattern_counter[
                pattern
            ] += 1

            file_pattern_counter[
                pattern
            ] += 1

        # ----------------------------------------------------
        # Store requirement-level info
        # ----------------------------------------------------

        requirement_statistics[
            json_file.name
        ].append({

            "requirement_id":
                requirement_id,

            "patterns":
                valid_patterns
        })

    # --------------------------------------------------------
    # STORE FILE-LEVEL STATS
    # --------------------------------------------------------

    file_statistics[json_file.name] = {

        "violations":
            file_violation_count,

        "pattern_distribution":
            dict(file_pattern_counter)
    }

# ============================================================
# PRINT GLOBAL RESULTS
# ============================================================

print("\n" + "=" * 70)

print(
    "GLOBAL COMPLIANCE VIOLATION STATISTICS"
)

print("=" * 70)

print(
    f"\nTotal JSON files processed: "
    f"{total_files}"
)

print(
    f"Total violations found:     "
    f"{total_violations}"
)

print("\nGLOBAL PATTERN DISTRIBUTION")

print("-" * 70)

for pattern in sorted(ALLOWED_PATTERNS):

    count = global_pattern_counter.get(
        pattern,
        0
    )

    percentage = (
        (count / total_violations) * 100
        if total_violations > 0 else 0
    )

    print(
        f"{pattern:<35} "
        f"{count:>6} "
        f"({percentage:.2f}%)"
    )

# ============================================================
# PRINT FILE-LEVEL SUMMARY
# ============================================================

print("\nFILE-LEVEL STATISTICS")

print("-" * 70)

for file_name, stats in file_statistics.items():

    print(f"\n{file_name}")

    print(
        f"  Violations: "
        f"{stats['violations']}"
    )

    if stats["pattern_distribution"]:

        for pattern, count in sorted(
            stats[
                "pattern_distribution"
            ].items()
        ):

            print(
                f"    {pattern:<30} "
                f"{count}"
            )

    else:

        print(
            "    No recognized patterns found"
        )

# ============================================================
# SAVE RESULTS
# ============================================================

summary = {

    "total_files":
        total_files,

    "total_violations":
        total_violations,

    "global_pattern_distribution":
        dict(global_pattern_counter),

    "file_statistics":
        file_statistics,

    "requirement_statistics":
        dict(requirement_statistics)
}

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)

print(
    f"Statistics saved to:\n"
    f"{OUTPUT_JSON}"
)

print("=" * 70)

In [ ]:
# ===============================================================================
# PRELIMINARY STEP: SIMPLIFY PTS FOR RESOLVING COMPLIANCE VIOLATIONS WITH AN LLM
# ===============================================================================
#
# This script converts original CPEE process models (.xml)
# into simplified Process Structure Trees (PSTs) represented
# as readable text. The generated simplified PSTs are later
# used as input for the automated generation of resolution
# strategies.
#
# The script supports executing either:
# - one selected scenario
# - all available scenarios
#
# Output:
# - simplified PST text files
#
# ==============================================================================

import sys
import json
from pathlib import Path
import tempfile
import xml.etree.ElementTree as ET
import subprocess

# -----------------------------------
# Configuration
# -----------------------------------

RUN_ALL_SCENARIOS = False
SELECTED_SCENARIO = "01_awad_delivery_of_goods"

SCENARIOS = [
    "01_awad_delivery_of_goods",
    "02_de_masellis_loan_approval",
    "03_elgammal_loan_approval",
    "04_ghose_package_screening",
    "05_loebbecke_blood_collection",
    "06_BPMQ",
    "07_CRL",
    "08_DCR",
    "09_HaarmannetAL2021",
    "11_PCL",
    "12_PPSL",
    "13_Status",
    "14_SunetAl24",
    "15_WinteretAlER",
    "16_Zasadaetal23"
]

# -----------------------------------
# Dynamically locate project root
# -----------------------------------

BASE_DIR = Path.cwd().parents[1]

# -----------------------------------
# Add paths
# -----------------------------------

SRC_DIR = BASE_DIR / "src"
sys.path.append(str(SRC_DIR))

STEP1_DIR = SRC_DIR / "step_1_generate_resolution_strategies"
sys.path.append(str(STEP1_DIR))

from utils.xml_loader import load_xml
from converter.cpee_to_simplified_pst import convert
from utils.exporter import pst_to_dict, pst_to_text

# -----------------------------------
# Directories
# -----------------------------------

PST_DIR = BASE_DIR / "data" / "input" / "process_models"

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "simplified_pst"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------
# Run one scenario
# -----------------------------------

def run_scenario(scenario_name):
    print("\n===================================")
    print(f"RUNNING SCENARIO: {scenario_name}")
    print("===================================\n")

    xml_file = PST_DIR / f"{scenario_name}.xml"

    print(f"Using XML file: {xml_file}")

    root = load_xml(xml_file)
    pst = convert(root)

    pst_text = pst_to_text(pst)

    if not pst_text.rstrip().endswith("terminate"):
        pst_text = pst_text.rstrip() + "\nterminate"

    print("\n=== PST (Tree) ===")
    print(pst_text)

    print("\n=== PST (JSON) ===")
    print(
        json.dumps(
            pst_to_dict(pst),
            indent=2
        )
    )

    output_file = (
        OUTPUT_DIR
        / f"{scenario_name}_simplified_pst.txt"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(pst_text)

    print(
        f"\nSimplified PST saved to: "
        f"{output_file}"
    )

# -----------------------------------
# Main execution
# -----------------------------------

if RUN_ALL_SCENARIOS:
    for scenario in SCENARIOS:
        run_scenario(scenario)

else:
    run_scenario(SELECTED_SCENARIO)

In [ ]:
# ============================================================
# STEP 1:
# GENERATE RESOLUTION STRATEGIES
# ============================================================
#
# This script generates candidate resolution strategies for
# detected compliance violations using a Large Language Model
# (LLM).
#
# For a selected scenario and violated requirement, the script:
# - loads the simplified Process Structure Tree (PST)
# - loads the detected compliance violation
# - constructs a prompting context
# - queries the LLM for candidate repair strategies
# - stores the generated strategies and metadata
#
# The generated resolution strategies consist of sequences of
# change operations intended to mitigate or resolve the
# detected compliance violation.
#
# Output:
# - generated resolution strategies (.json)
# - raw LLM responses
# - metadata and execution statistics
#
# ============================================================

import json
import time
import requests
from pathlib import Path

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# CONFIGURATION
# ============================================================

SCENARIO_NAME = "16_Zasadaetal23"

REQUIREMENT_ID = "R3"

MODEL = "openai/gpt-5.5"

# ============================================================
# LOAD API KEY
# ============================================================

API_KEY_FILE = (
    BASE_DIR
    / "config"
    / "api_keys.json"
)

with open(
    API_KEY_FILE,
    "r",
    encoding="utf-8"
) as f:

    api_config = json.load(f)

OPENROUTER_API_KEY = (
    api_config["OPENROUTER_API_KEY"]
)

# ============================================================
# INPUT FILES
# ============================================================

PROMPT_FILE = (
    BASE_DIR
    / "data"
    / "input"
    / "prompts"
    / "generate_resolution_strategies.txt"
)

PST_FILE = (
    BASE_DIR
    / "data"
    / "output"
    / "simplified_pst"
    / f"{SCENARIO_NAME}_simplified_pst.txt"
)

VIOLATIONS_FILE = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_before_changes"
    / f"{SCENARIO_NAME}_ALL_violations.json"
)

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "generated_resolution_strategies"
)

SCENARIO_OUTPUT_DIR = (
    OUTPUT_DIR
    / SCENARIO_NAME
)

JSON_DIR = (
    SCENARIO_OUTPUT_DIR
    / "resolution_strategies_clean"
)

RAW_DIR = (
    SCENARIO_OUTPUT_DIR
    / "raw"
)

FULL_RESPONSE_DIR = (
    SCENARIO_OUTPUT_DIR
    / "full_response"
)

METADATA_DIR = (
    SCENARIO_OUTPUT_DIR
    / "metadata"
)

PROBLEMS_DIR = (
    SCENARIO_OUTPUT_DIR
    / "problems"
)

JSON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FULL_RESPONSE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROBLEMS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# TIMER START
# ============================================================

start_time = time.time()

# ============================================================
# LOGGING VARIABLES
# ============================================================

problems_detected = []

# ============================================================
# LOAD FILES
# ============================================================

with open(
    PROMPT_FILE,
    "r",
    encoding="utf-8"
) as f:

    BASE_PROMPT = f.read()

with open(
    PST_FILE,
    "r",
    encoding="utf-8"
) as f:

    pst = f.read()

with open(
    VIOLATIONS_FILE,
    "r",
    encoding="utf-8"
) as f:

    violations = json.load(f)

# ============================================================
# FIND TARGET VIOLATION
# ============================================================

target_violation = None

for violation in violations:

    if violation["requirement_id"] == REQUIREMENT_ID:

        target_violation = violation

        break

if target_violation is None:

    raise ValueError(
        f"Violation '{REQUIREMENT_ID}' not found."
    )

print(
    f"Using violation: "
    f"{REQUIREMENT_ID}"
)

# ============================================================
# BUILD OUTPUT PREFIX
# ============================================================

OUTPUT_PREFIX = (
    f"{SCENARIO_NAME}_RS_{REQUIREMENT_ID}"
)

print(
    "Output prefix:",
    OUTPUT_PREFIX
)

# ============================================================
# BUILD PROMPT
# ============================================================

final_prompt = f"""{BASE_PROMPT}

============================================================
PROCESS STRUCTURED TREE
============================================================

{pst}

============================================================
DETECTED VIOLATION
============================================================

{json.dumps(target_violation, indent=2)}

IMPORTANT:
- Return ONLY valid JSON
- Do not use markdown
- Do not use code fences
- Ensure output is parseable with json.loads()
"""

# ============================================================
# API REQUEST
# ============================================================

print(
    "\nSending request to model...\n"
)

try:

    response = requests.post(
        url=(
            "https://openrouter.ai/"
            "api/v1/chat/completions"
        ),
        headers={
            "Authorization": (
                f"Bearer "
                f"{OPENROUTER_API_KEY}"
            ),
            "Content-Type":
                "application/json",
        },
        data=json.dumps({
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": final_prompt
                }
            ],
            "temperature": 0,
            "reasoning": {
                "enabled": True
            }
        }),
        timeout=300
    )

except Exception as e:

    problems_detected.append(
        f"Request exception: {str(e)}"
    )

    raise e

# ============================================================
# CHECK RESPONSE
# ============================================================

if response.status_code != 200:

    problems_detected.append(
        f"HTTP {response.status_code}"
    )

    print("ERROR:")

    print(response.status_code)

    print(response.text)

    raise Exception(
        "API request failed"
    )

response_json = response.json()

# ============================================================
# EXTRACT MODEL OUTPUT
# ============================================================

print(
    "\n========== FULL RESPONSE ==========\n"
)

print(
    json.dumps(
        response_json,
        indent=2
    )
)

if "choices" not in response_json:

    problems_detected.append(
        "API response missing 'choices'"
    )

    raise ValueError(
        "API response does not contain "
        "'choices'. See printed response above."
    )

try:

    assistant_message = (
        response_json["choices"][0]["message"]
    )

    generated_text = (
        assistant_message["content"]
    )

except Exception as e:

    problems_detected.append(
        "Failed to extract assistant message"
    )

    raise e

# ============================================================
# VALIDATE JSON
# ============================================================

parsed_json = None

try:

    parsed_json = json.loads(
        generated_text
    )

except Exception as e:

    problems_detected.append(
        "Model output is not valid JSON"
    )

    print(
        "\nERROR: Invalid JSON generated"
    )

    print(e)

# ============================================================
# SAVE RAW RESPONSE
# ============================================================

raw_output_path = (
    RAW_DIR
    / f"{OUTPUT_PREFIX}_raw.txt"
)

with open(
    raw_output_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(generated_text)

print(
    f"\nRaw response saved to:\n"
    f"{raw_output_path}"
)

# ============================================================
# SAVE PARSED JSON
# ============================================================

if parsed_json is not None:

    json_output_path = (
        JSON_DIR
        / f"{OUTPUT_PREFIX}.json"
    )

    with open(
        json_output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed_json,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nJSON saved to:\n"
        f"{json_output_path}"
    )

# ============================================================
# SAVE FULL API RESPONSE
# ============================================================

full_response_path = (
    FULL_RESPONSE_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_full_response.json"
    )
)

with open(
    full_response_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        response_json,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nFull API response saved to:\n"
    f"{full_response_path}"
)

# ============================================================
# EXECUTION TIME
# ============================================================

end_time = time.time()

execution_time_ms = round(
    (end_time - start_time) * 1000,
    2
)

# ============================================================
# TOKEN USAGE
# ============================================================

usage = response_json.get(
    "usage",
    {}
)

# ============================================================
# SAVE METADATA
# ============================================================

metadata = {

    "model":
        MODEL,

    "scenario_name":
        SCENARIO_NAME,

    "requirement_id":
        REQUIREMENT_ID,

    "output_prefix":
        OUTPUT_PREFIX,

    "execution_time_milliseconds":
        execution_time_ms,

    "problems_detected":
        problems_detected,

    "usage": {

        "prompt_tokens":
            usage.get(
                "prompt_tokens"
            ),

        "completion_tokens":
            usage.get(
                "completion_tokens"
            ),

        "total_tokens":
            usage.get(
                "total_tokens"
            ),

        "reasoning_tokens":
            usage.get(
                "reasoning_tokens"
            )
    }
}

metadata_path = (
    METADATA_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_metadata.json"
    )
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nMetadata saved to:\n"
    f"{metadata_path}"
)

# ============================================================
# SAVE PROBLEM LOG
# ============================================================

problem_log_path = (
    PROBLEMS_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_problems.log"
    )
)

with open(
    problem_log_path,
    "w",
    encoding="utf-8"
) as f:

    if len(problems_detected) == 0:

        f.write(
            "No problems detected.\n"
        )

    else:

        for problem in problems_detected:

            f.write(problem + "\n")

print(
    f"\nProblem log saved to:\n"
    f"{problem_log_path}"
)

# ============================================================
# SUMMARY
# ============================================================

print(
    "\n========== EXECUTION SUMMARY ==========\n"
)

print(
    "Execution time (milliseconds):",
    execution_time_ms
)

print(
    "Prompt tokens:",
    usage.get("prompt_tokens")
)

print(
    "Completion tokens:",
    usage.get("completion_tokens")
)

print(
    "Total tokens:",
    usage.get("total_tokens")
)

print(
    "Reasoning tokens:",
    usage.get("reasoning_tokens")
)

print("\nProblems detected:")

if len(problems_detected) == 0:

    print("None")

else:

    for problem in problems_detected:

        print("-", problem)

In [ ]:
# old
# ============================================================
# STEP 2:
# APPLY AND VALIDATE RESOLUTION STRATEGIES
# ============================================================
#
# This script applies the generated resolution strategies to
# the original process models and validates the resulting
# updated Process Structure Trees (PSTs).
#
# For each scenario and violated requirement, the script:
# - loads the original process model
# - loads the generated resolution strategies
# - applies each change operation sequentially
# - validates the updated process structure
# - stores the updated PSTs and execution logs
#
# The validation pipeline includes:
# - change operation validation
# - behavioral validation
# - PST validation
# - structural validation
#
# The script supports executing either:
# - one selected scenario/requirement
# - all configured scenarios/requirements
#
# Output:
# - updated PST models (.xml)
# - validation and execution logs
#
# ============================================================

import sys
import json
import time
from pathlib import Path

# ============================================================
# EXECUTION MODE
# ============================================================

RUN_ALL = True

SCENARIO_NAME = "16_Zasadaetal23"

REQUIREMENT_ID = "R3"

# ============================================================
# SCENARIO / REQUIREMENT CONFIGURATION
# ============================================================

SCENARIO_REQUIREMENTS = {

    "03_elgammal_loan_approval": [
        "R2",
        "R3"
    ],

    "13_Status": [
        "R1",
        "R2"
    ],

    "11_PCL": [
        "R1",
        "R3",
        "R4"
    ],

    "14_SunetAl24": [
        "R2",
        "R3",
        "R4"
    ],

    "16_Zasadaetal23": [
        "R2",
        "R3",
        "R5"
    ],

    "07_CRL": [
        "R1",
        "R3",
        "R4",
        "R6"
    ],

    "06_BPMQ": [
        "R2",
        "R4",
        "R5",
        "R6"
    ],

    "12_PPSL": [
        "R2",
        "R3",
        "R4"
    ],

    "02_de_masellis_loan_approval": [
        "R1",
        "R2",
        "R3"
    ],

    "04_ghose_package_screening": [
        "R1"
    ],

    "08_DCR": [
        "R2"
    ],

    "05_loebbecke_blood_collection": [
        "R6"
    ],

    "01_awad_delivery_of_goods": [
        "R1",
        "R2",
        "R3",
        "R4"
    ],

    "09_HaarmannetAL2021": [
        "R1"
    ],

    "15_WinteretAlER": [
        "R2",
        "R3"
    ]
}

# ============================================================
# BUILD EXECUTION LIST
# ============================================================

EXECUTION_LIST = []

if RUN_ALL:

    for scenario, requirements in (
        SCENARIO_REQUIREMENTS.items()
    ):

        for req_id in requirements:

            EXECUTION_LIST.append(
                (scenario, req_id)
            )

else:

    EXECUTION_LIST.append(
        (
            SCENARIO_NAME,
            REQUIREMENT_ID
        )
    )

# ============================================================
# PYTHON PATHS
# ============================================================

from pathlib import Path
import sys

# Current file directory
CURRENT_DIR = Path(__file__).resolve().parent

# Project src root
SRC_DIR = CURRENT_DIR.parent

# step_2_apply_validate_resolution_strategies directory
STEP2_DIR = SRC_DIR / "step_2_apply_validate_resolution_strategies"

# Add to Python path
sys.path.append(str(SRC_DIR))
sys.path.append(str(STEP2_DIR))

# ============================================================
# IMPORTS
# ============================================================

from utils.process_io import load_process, save_process

from change_management.change_applier import (
    ChangeApplier
)

from validators.change_operation_validator import (
    ChangeOperationValidator
)

from validators.behavioral_validator import (
    BehavioralValidator
)

from validators.pst_validator import (
    PSTValidator
)

from validators.structural_validator import (
    StructuralValidator
)

from change_operations.operations import *

# ============================================================
# BASE PATH
# ============================================================

from pathlib import Path

# Project root directory
BASE_PATH = Path(__file__).resolve().parent.parent

# ============================================================
# CHANGE OPERATION MAPPING
# ============================================================

operation_mapping = {
    "insert_after": insert_after,
    "insert_before": insert_before,
    "delete": delete,
    "rename": rename,
    "move_after": move_after,
    "move_before": move_before,
    "swap": swap,
    "merge": merge_by_label,
    "split": split,
    "copy_after": copy_after,
    "copy_before": copy_before,
    "modify_condition": modify_condition,
    "modify_resource": modify_resource,
    "modify_write": modify_write,
    "add_write": add_write,
    "remove_write": remove_write,
    "modify_read": modify_read,
    "parallelize": parallelize,
    "sequentialize_parallel": sequentialize_parallel,
    "add_xor_branch": add_xor_branch,
    "remove_branch": remove_branch,
    "remove_branch_by_condition":
        remove_branch_by_condition,
    "embed_activity_in_xor":
        embed_activity_in_xor,
    "embed_pre_loop": embed_pre_loop,
    "embed_post_loop": embed_post_loop,
    "remove_loop": remove_loop,
    "modify_loop_condition":
        modify_loop_condition,
    "modify_timeout": modify_timeout
}

# ============================================================
# VALIDATORS + APPLIER
# ============================================================

applier = ChangeApplier()

pattern_validator = ChangeOperationValidator()

behavioral_validator = BehavioralValidator()

pst_validator = PSTValidator()

structural_validator = StructuralValidator()

# ============================================================
# EXECUTION
# ============================================================

for (
    SCENARIO_NAME,
    REQUIREMENT_ID
) in EXECUTION_LIST:

    print("\n================================================")
    print("SCENARIO:", SCENARIO_NAME)
    print("REQUIREMENT:", REQUIREMENT_ID)
    print("================================================")

    # --------------------------------------------------------
    # INPUT FILES
    # --------------------------------------------------------

    ORIGINAL_MODEL_PATH = (
        BASE_PATH
        / "data/input/original_pst"
        / f"{SCENARIO_NAME}.xml"
    )

    RESOLUTION_STRATEGIES_FILE = (
        BASE_PATH
        / "data/output/generated_resolution_strategies"
        / SCENARIO_NAME
        / "resolution_strategies_clean"
        / f"{SCENARIO_NAME}_RS_{REQUIREMENT_ID}.json"
    )

    # --------------------------------------------------------
    # SKIP MISSING FILES
    # --------------------------------------------------------

    if not RESOLUTION_STRATEGIES_FILE.exists():

        print(
            "Resolution strategy file missing:"
        )

        print(RESOLUTION_STRATEGIES_FILE)

        continue

    # --------------------------------------------------------
    # OUTPUT DIRECTORIES
    # --------------------------------------------------------

    SCENARIO_OUTPUT_DIR = (
        BASE_PATH
        / "data/output/updated_pst"
        / SCENARIO_NAME
    )

    PST_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "pst"
    )

    LOG_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "logs"
    )

    PST_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    LOG_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # LOAD RESOLUTION STRATEGIES
    # --------------------------------------------------------

    with open(
        RESOLUTION_STRATEGIES_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        resolution_data = json.load(f)

    resolution_strategies = (
        resolution_data[
            "resolution_strategies"
        ]
    )

    # --------------------------------------------------------
    # APPLY EACH STRATEGY
    # --------------------------------------------------------

    for strategy in resolution_strategies:

        strategy_id = strategy[
            "resolution_strategy_id"
        ]

        print("\n------------------------------------------------")
        print("Applying strategy:", strategy_id)
        print("------------------------------------------------")

        tree, root = load_process(
            str(ORIGINAL_MODEL_PATH)
        )

        current_root = root

        strategy_logs = []

        strategy_start_time = time.time()

        # ----------------------------------------------------
        # APPLY OPERATIONS
        # ----------------------------------------------------

        for operation_data in strategy[
            "change_operations"
        ]:

            operation_name = operation_data[
                "operation"
            ]

            parameters = operation_data[
                "parameters"
            ]

            print(
                "\nOperation:",
                operation_name
            )

            print(
                "Parameters:",
                parameters
            )

            if (
                operation_name
                not in operation_mapping
            ):

                raise ValueError(
                    f"Unsupported operation: "
                    f"{operation_name}"
                )

            operation_function = (
                operation_mapping[
                    operation_name
                ]
            )

            # ------------------------------------------------
            # VALIDATE OPERATION
            # ------------------------------------------------

            pattern_validator.validate(
                operation_function
            )

            strategy_logs.append(
                f"ChangeOperationValidator: "
                f"SUCCESS ({operation_name})"
            )

            # ------------------------------------------------
            # ARGUMENT MAPPING
            # ------------------------------------------------

            args = []

            if operation_name in [
                "insert_after",
                "insert_before"
            ]:

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == "delete":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "rename":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name in [
                "move_after",
                "move_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "swap":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "merge":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

                if "keep_position" in parameters:

                    args.append(
                        parameters[
                            "keep_position"
                        ]
                    )

            elif operation_name == "split":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "copy_after",
                "copy_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "parallelize",
                "sequentialize_parallel"
            ]:

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "remove_branch":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "remove_branch_by_condition"
            ):

                args = [
                    parameters[
                        "target_condition"
                    ]
                ]

            elif operation_name == (
                "modify_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_resource"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_resource"
                    ]
                ]

            elif operation_name == (
                "modify_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_statement"
                    ]
                ]

            elif operation_name == "add_write":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "added_statement"
                    ]
                ]

            elif operation_name == (
                "remove_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "removed_variable_name"
                    ]
                ]

            elif operation_name == (
                "modify_read"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "old_variable_name"
                    ],
                    parameters[
                        "new_variable_name"
                    ]
                ]

            elif operation_name == "remove_loop":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "modify_loop_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_timeout"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_timeout"
                    ]
                ]

            elif operation_name == (
                "add_xor_branch"
            ):

                args = [
                    parameters[
                        "existing_branch_condition"
                    ],
                    parameters[
                        "new_branch_condition"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == (
                "embed_activity_in_xor"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "condition"
                    ],
                    parameters[
                        "mode"
                    ]
                ]

            elif operation_name in [
                "embed_pre_loop",
                "embed_post_loop"
            ]:

                args = [
                    parameters[
                        "start_activity_label"
                    ],
                    parameters[
                        "end_activity_label"
                    ],
                    parameters[
                        "condition"
                    ]
                ]

            else:

                raise ValueError(
                    f"Argument mapping missing "
                    f"for operation: "
                    f"{operation_name}"
                )

            # ------------------------------------------------
            # APPLY CHANGE
            # ------------------------------------------------

            try:

                current_root, log = (
                    applier.apply(
                        current_root,
                        operation_function,
                        *args
                    )
                )

                strategy_logs.append(log)

                behavioral_validator.validate(
                    current_root
                )

                strategy_logs.append(
                    "BehavioralValidator: SUCCESS"
                )

                pst_validator.validate(
                    current_root
                )

                strategy_logs.append(
                    "PSTValidator: SUCCESS"
                )

                structural_warnings = (
                    structural_validator.validate(
                        current_root
                    )
                )

                strategy_logs.append(
                    "StructuralValidator: SUCCESS"
                )

                if structural_warnings:

                    for warning in (
                        structural_warnings
                    ):

                        strategy_logs.append(
                            warning
                        )

                print("SUCCESS")

            except Exception as e:

                print("FAILED")

                print(str(e))

                strategy_logs.append(
                    f"FAILED: {str(e)}"
                )

                break

        # ----------------------------------------------------
        # EXECUTION TIME
        # ----------------------------------------------------

        strategy_end_time = time.time()

        execution_time_ms = round(
            (
                strategy_end_time
                - strategy_start_time
            ) * 1000,
            2
        )

        strategy_logs.append(
            f"Execution time "
            f"(milliseconds): "
            f"{execution_time_ms}"
        )

        # ----------------------------------------------------
        # SAVE UPDATED MODEL
        # ----------------------------------------------------

        tree._setroot(current_root)

        output_model_path = (
            PST_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}.xml"
            )
        )

        save_process(
            tree,
            str(output_model_path)
        )

        print("\nSaved updated model:")

        print(output_model_path)

        # ----------------------------------------------------
        # SAVE LOG
        # ----------------------------------------------------

        log_path = (
            LOG_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}_log.txt"
            )
        )

        with open(
            log_path,
            "w",
            encoding="utf-8"
        ) as f:

            for log_entry in strategy_logs:

                f.write(str(log_entry))

                f.write("\n")

        print("Saved log:")

        print(log_path)

print("\n================================================")
print("ALL STRATEGIES COMPLETED")
print("================================================")

In [ ]:
# ============================================================
# STEP 2:
# APPLY AND VALIDATE RESOLUTION STRATEGIES
# ============================================================
#
# This script applies the generated resolution strategies to
# the original process models and validates the resulting
# updated Process Structure Trees (PSTs).
#
# For each scenario and violated requirement, the script:
# - loads the original process model
# - loads the generated resolution strategies
# - applies each change operation sequentially
# - validates the updated process structure
# - stores the updated PSTs and execution logs
#
# The validation pipeline includes:
# - change operation validation
# - behavioral validation
# - PST validation
# - structural validation
#
# The script also generates:
# - a global CSV summary
# - validation statistics by scenario
#
# Output:
# - updated PST models (.xml)
# - validation and execution logs
# - validation summary CSV files
#
# ============================================================


# ============================================================
# STEP 2:
# APPLY AND VALIDATE RESOLUTION STRATEGIES
# ============================================================

import sys
import json
import time
import csv
from pathlib import Path
from collections import defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_PATH = CURRENT_DIR

while BASE_PATH.name != (
    "PTResolver"
):
    BASE_PATH = BASE_PATH.parent

print("Project root:")
print(BASE_PATH)

# ============================================================
# EXECUTION MODE
# ============================================================

RUN_ALL = True

SCENARIO_NAME = "16_Zasadaetal23"

REQUIREMENT_ID = "R3"

# ============================================================
# SCENARIO / REQUIREMENT CONFIGURATION
# ============================================================

SCENARIO_REQUIREMENTS = {

    "03_elgammal_loan_approval": [
        "R2",
        "R3"
    ],

    "13_Status": [
        "R1",
        "R2"
    ],

    "11_PCL": [
        "R1",
        "R3",
        "R4"
    ],

    "14_SunetAl24": [
        "R2",
        "R3",
        "R4"
    ],

    "16_Zasadaetal23": [
        "R2",
        "R3",
        "R5"
    ],

    "07_CRL": [
        "R1",
        "R3",
        "R4",
        "R6"
    ],

    "06_BPMQ": [
        "R2",
        "R4",
        "R5",
        "R6"
    ],

    "12_PPSL": [
        "R2",
        "R3",
        "R4"
    ],

    "02_de_masellis_loan_approval": [
        "R1",
        "R2",
        "R3"
    ],

    "04_ghose_package_screening": [
        "R1"
    ],

    "08_DCR": [
        "R2"
    ],

    "05_loebbecke_blood_collection": [
        "R6"
    ],

    "01_awad_delivery_of_goods": [
        "R1",
        "R2",
        "R3",
        "R4"
    ],

    "09_HaarmannetAL2021": [
        "R1"
    ],

    "15_WinteretAlER": [
        "R2",
        "R3"
    ]
}

# ============================================================
# BUILD EXECUTION LIST
# ============================================================

EXECUTION_LIST = []

if RUN_ALL:

    for scenario, requirements in (
        SCENARIO_REQUIREMENTS.items()
    ):

        for req_id in requirements:

            EXECUTION_LIST.append(
                (scenario, req_id)
            )

else:

    EXECUTION_LIST.append(
        (
            SCENARIO_NAME,
            REQUIREMENT_ID
        )
    )

# ============================================================
# PYTHON PATHS
# ============================================================

SRC_DIR = (
    BASE_PATH
    / "src"
)

STEP2_DIR = (
    SRC_DIR
    / "step_2_apply_validate_resolution_strategies"
)

sys.path.append(str(SRC_DIR))
sys.path.append(str(STEP2_DIR))

# ============================================================
# IMPORTS
# ============================================================

from utils.process_io import load_process, save_process

from change_management.change_applier import (
    ChangeApplier
)

from validators.change_operation_validator import (
    ChangeOperationValidator
)

from validators.behavioral_validator import (
    BehavioralValidator
)

from validators.pst_validator import (
    PSTValidator
)

from validators.structural_validator import (
    StructuralValidator
)

from change_operations.operations import *

# ============================================================
# CHANGE OPERATION MAPPING
# ============================================================

operation_mapping = {
    "insert_after": insert_after,
    "insert_before": insert_before,
    "delete": delete,
    "rename": rename,
    "move_after": move_after,
    "move_before": move_before,
    "swap": swap,
    "merge": merge_by_label,
    "split": split,
    "copy_after": copy_after,
    "copy_before": copy_before,
    "modify_condition": modify_condition,
    "modify_resource": modify_resource,
    "modify_write": modify_write,
    "add_write": add_write,
    "remove_write": remove_write,
    "modify_read": modify_read,
    "parallelize": parallelize,
    "sequentialize_parallel": sequentialize_parallel,
    "add_xor_branch": add_xor_branch,
    "remove_branch": remove_branch,
    "remove_branch_by_condition":
        remove_branch_by_condition,
    "embed_activity_in_xor":
        embed_activity_in_xor,
    "embed_pre_loop": embed_pre_loop,
    "embed_post_loop": embed_post_loop,
    "remove_loop": remove_loop,
    "modify_loop_condition":
        modify_loop_condition,
    "modify_timeout": modify_timeout
}

# ============================================================
# VALIDATORS + APPLIER
# ============================================================

applier = ChangeApplier()

pattern_validator = ChangeOperationValidator()

behavioral_validator = BehavioralValidator()

pst_validator = PSTValidator()

structural_validator = StructuralValidator()

# ============================================================
# GLOBAL VALIDATION SUMMARY
# ============================================================

validation_summary = []

scenario_statistics = defaultdict(
    lambda: {
        "total_strategies": 0,
        "applied": 0,
        "warnings": 0,
        "errors": 0
    }
)

# ============================================================
# EXECUTION
# ============================================================

for (
    SCENARIO_NAME,
    REQUIREMENT_ID
) in EXECUTION_LIST:

    print("\n================================================")
    print("SCENARIO:", SCENARIO_NAME)
    print("REQUIREMENT:", REQUIREMENT_ID)
    print("================================================")

    ORIGINAL_MODEL_PATH = (
        BASE_PATH
        / "data/input/original_pst"
        / f"{SCENARIO_NAME}.xml"
    )

    RESOLUTION_STRATEGIES_FILE = (
        BASE_PATH
        / "data/output/generated_resolution_strategies"
        / SCENARIO_NAME
        / "resolution_strategies_clean"
        / f"{SCENARIO_NAME}_RS_{REQUIREMENT_ID}.json"
    )

    if not RESOLUTION_STRATEGIES_FILE.exists():

        print(
            "Resolution strategy file missing:"
        )

        print(RESOLUTION_STRATEGIES_FILE)

        continue

    SCENARIO_OUTPUT_DIR = (
        BASE_PATH
        / "data/output/updated_pst"
        / SCENARIO_NAME
    )

    PST_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "pst"
    )

    LOG_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "logs"
    )

    PST_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    LOG_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        RESOLUTION_STRATEGIES_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        resolution_data = json.load(f)

    resolution_strategies = (
        resolution_data[
            "resolution_strategies"
        ]
    )

    for strategy in resolution_strategies:

        strategy_id = strategy[
            "resolution_strategy_id"
        ]

        print("\n------------------------------------------------")
        print("Applying strategy:", strategy_id)
        print("------------------------------------------------")

        tree, root = load_process(
            str(ORIGINAL_MODEL_PATH)
        )

        current_root = root

        strategy_logs = []

        applied_operations = []

        strategy_start_time = time.time()

        strategy_applied = "YES"

        validation_status = "SUCCESS"

        warnings_detected = "NO"

        change_operation_validation = (
            "SUCCESS"
        )

        behavioral_validation = (
            "NOT_EXECUTED"
        )

        pst_validation = (
            "NOT_EXECUTED"
        )

        structural_validation = (
            "NOT_EXECUTED"
        )

        failure_message = ""

        for operation_data in strategy[
            "change_operations"
        ]:

            operation_name = operation_data[
                "operation"
            ]

            parameters = operation_data[
                "parameters"
            ]

            print(
                "\nOperation:",
                operation_name
            )

            print(
                "Parameters:",
                parameters
            )

            if (
                operation_name
                not in operation_mapping
            ):

                raise ValueError(
                    f"Unsupported operation: "
                    f"{operation_name}"
                )

            operation_function = (
                operation_mapping[
                    operation_name
                ]
            )

            pattern_validator.validate(
                operation_function
            )

            strategy_logs.append(
                f"ChangeOperationValidator: "
                f"SUCCESS ({operation_name})"
            )

            args = []

            # =================================================
            # ARGUMENT MAPPING
            # =================================================

            if operation_name in [
                "insert_after",
                "insert_before"
            ]:

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == "delete":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "rename":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name in [
                "move_after",
                "move_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "swap":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "merge":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "split":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "copy_after",
                "copy_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "parallelize",
                "sequentialize_parallel"
            ]:

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "remove_branch":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "remove_branch_by_condition"
            ):

                args = [
                    parameters[
                        "target_condition"
                    ]
                ]

            elif operation_name == (
                "modify_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_resource"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_resource"
                    ]
                ]

            elif operation_name == (
                "modify_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_statement"
                    ]
                ]

            elif operation_name == "add_write":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "added_statement"
                    ]
                ]

            elif operation_name == (
                "remove_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "removed_variable_name"
                    ]
                ]

            elif operation_name == (
                "modify_read"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "old_variable_name"
                    ],
                    parameters[
                        "new_variable_name"
                    ]
                ]

            elif operation_name == "remove_loop":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "modify_loop_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_timeout"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_timeout"
                    ]
                ]

            elif operation_name == (
                "add_xor_branch"
            ):

                args = [
                    parameters[
                        "existing_branch_condition"
                    ],
                    parameters[
                        "new_branch_condition"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == (
                "embed_activity_in_xor"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "condition"
                    ],
                    parameters[
                        "mode"
                    ]
                ]

            elif operation_name in [
                "embed_pre_loop",
                "embed_post_loop"
            ]:

                args = [
                    parameters[
                        "start_activity_label"
                    ],
                    parameters[
                        "end_activity_label"
                    ],
                    parameters[
                        "condition"
                    ]
                ]

            else:

                raise ValueError(
                    f"Argument mapping missing "
                    f"for operation: "
                    f"{operation_name}"
                )

            # =================================================
            # APPLY CHANGE
            # =================================================

            try:

                current_root, log = (
                    applier.apply(
                        current_root,
                        operation_function,
                        *args
                    )
                )

                applied_operations.append(
                    f"{operation_name}"
                )

                strategy_logs.append(log)

            except Exception as e:

                print("FAILED TO APPLY CHANGE")

                print(str(e))

                strategy_applied = "NO"

                validation_status = "ERROR"

                failure_message = str(e)

                strategy_logs.append(
                    f"APPLICATION FAILED: {str(e)}"
                )

                break

            # =================================================
            # VALIDATION PHASE
            # =================================================

            try:

                behavioral_validator.validate(
                    current_root
                )

                behavioral_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "BehavioralValidator: SUCCESS"
                )

            except Exception as e:

                behavioral_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"BehavioralValidator WARNING: "
                    f"{str(e)}"
                )

            try:

                pst_validator.validate(
                    current_root
                )

                pst_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "PSTValidator: SUCCESS"
                )

            except Exception as e:

                pst_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"PSTValidator WARNING: "
                    f"{str(e)}"
                )

            try:

                structural_warnings = (
                    structural_validator.validate(
                        current_root
                    )
                )

                structural_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "StructuralValidator: SUCCESS"
                )

                if structural_warnings:

                    warnings_detected = (
                        "YES"
                    )

                    validation_status = (
                        "WARNING"
                    )

                    for warning in (
                        structural_warnings
                    ):

                        strategy_logs.append(
                            f"Structural WARNING: "
                            f"{warning}"
                        )

            except Exception as e:

                structural_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"StructuralValidator WARNING: "
                    f"{str(e)}"
                )

            print("SUCCESS")

        # =====================================================
        # EXECUTION TIME
        # =====================================================

        strategy_end_time = time.time()

        execution_time_ms = round(
            (
                strategy_end_time
                - strategy_start_time
            ) * 1000,
            2
        )

        strategy_logs.append(
            f"Execution time "
            f"(milliseconds): "
            f"{execution_time_ms}"
        )

        # =====================================================
        # SAVE VALIDATION SUMMARY
        # =====================================================

        validation_summary.append({

            "scenario":
                SCENARIO_NAME,

            "requirement_id":
                REQUIREMENT_ID,

            "strategy_id":
                strategy_id,

            "strategy_applied":
                strategy_applied,

            "validation_status":
                validation_status,

            "warnings_detected":
                warnings_detected,

            "change_operation_validator":
                change_operation_validation,

            "behavioral_validator":
                behavioral_validation,

            "pst_validator":
                pst_validation,

            "structural_validator":
                structural_validation,

            "applied_operations":
                " | ".join(
                    applied_operations
                ),

            "execution_time_ms":
                execution_time_ms,

            "failure_message":
                failure_message
        })

        # =====================================================
        # SCENARIO STATISTICS
        # =====================================================

        scenario_statistics[
            SCENARIO_NAME
        ]["total_strategies"] += 1

        if strategy_applied == "YES":

            scenario_statistics[
                SCENARIO_NAME
            ]["applied"] += 1

        if validation_status == "WARNING":

            scenario_statistics[
                SCENARIO_NAME
            ]["warnings"] += 1

        if validation_status == "ERROR":

            scenario_statistics[
                SCENARIO_NAME
            ]["errors"] += 1

        # =====================================================
        # SAVE UPDATED MODEL
        # =====================================================

        tree._setroot(current_root)

        output_model_path = (
            PST_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}.xml"
            )
        )

        save_process(
            tree,
            str(output_model_path)
        )

        print("\nSaved updated model:")

        print(output_model_path)

        # =====================================================
        # SAVE LOG
        # =====================================================

        log_path = (
            LOG_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}_log.txt"
            )
        )

        with open(
            log_path,
            "w",
            encoding="utf-8"
        ) as f:

            for log_entry in strategy_logs:

                f.write(str(log_entry))
                f.write("\n")

        print("Saved log:")

        print(log_path)

# ============================================================
# SAVE GLOBAL CSV SUMMARY
# ============================================================

summary_csv_path = (
    BASE_PATH
    / "data/output/updated_pst"
    / "validation_summary.csv"
)

with open(
    summary_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[

            "scenario",
            "requirement_id",
            "strategy_id",

            "strategy_applied",
            "validation_status",
            "warnings_detected",

            "change_operation_validator",
            "behavioral_validator",
            "pst_validator",
            "structural_validator",

            "applied_operations",

            "execution_time_ms",

            "failure_message"
        ]
    )

    writer.writeheader()

    for row in validation_summary:

        writer.writerow(row)

print("\nSaved global validation summary:")

print(summary_csv_path)

# ============================================================
# SAVE SCENARIO STATISTICS CSV
# ============================================================

scenario_stats_csv = (
    BASE_PATH
    / "data/output/updated_pst"
    / "scenario_statistics.csv"
)

with open(
    scenario_stats_csv,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[

            "scenario",

            "total_strategies",

            "applied",

            "warnings",

            "errors",

            "application_rate"
        ]
    )

    writer.writeheader()

    for (
        scenario,
        stats
    ) in scenario_statistics.items():

        application_rate = round(
            (
                stats["applied"]
                / stats["total_strategies"]
            ) * 100,
            2
        )

        writer.writerow({

            "scenario":
                scenario,

            "total_strategies":
                stats["total_strategies"],

            "applied":
                stats["applied"],

            "warnings":
                stats["warnings"],

            "errors":
                stats["errors"],

            "application_rate":
                application_rate
        })

print("\nSaved scenario statistics:")

print(scenario_stats_csv)

print("\n================================================")
print("ALL STRATEGIES COMPLETED")
print("================================================")

In [ ]:
# ============================================================
# STEP 3: VERIFY COMPLIANCE OF UPDATED PSTS
# ============================================================
#
# This script evaluates the updated Process Structure Trees
# (PSTs) generated after applying resolution strategies in
# order to determine whether the original compliance
# violations were successfully resolved.
#
# For each updated PST, the script:
# - injects the compliance requirement set
# - executes compliance verification using
#   ProcessTreeVerify
# - extracts the detected compliance violations
# - stores the resulting violation reports
#
# The script supports executing either:
# - one selected scenario
# - all available scenarios
#
# The generated violation reports are later used to analyze:
# - whether the targeted violations were fixed
# - which resolution strategies succeeded or failed
# - remaining compliance issues after adaptation
#
# Output:
# - compliance violation reports for updated PSTs (.json)
#
# ============================================================

import json
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

# -----------------------------------
# Configuration
# -----------------------------------

RUN_ALL_SCENARIOS = True

SELECTED_SCENARIO = (
    "01_awad_delivery_of_goods"
)

VERIFY_ALL = True

selected_requirement_ids = [
    "R1"
]

# -----------------------------------
# Dynamically locate project root
# -----------------------------------

BASE_DIR = Path.cwd().parents[1]

# -----------------------------------
# Paths
# -----------------------------------

PTV_SCRIPT = (
    BASE_DIR
    / "src"
    / "ProcessTreeVerify"
    / "python_code"
    / "test_script.py"
)

UPDATED_PST_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "updated_pst"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_after_changes"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------
# Namespaces
# -----------------------------------

NS_PROPERTIES = (
    "http://cpee.org/ns/properties/2.0"
)

NS_SUBSCRIPTIONS = (
    "http://riddl.org/ns/common-patterns/"
    "notifications-producer/2.0"
)

namespace = {
    "ns1": NS_PROPERTIES
}

# -----------------------------------
# Ensure <requirements> node
# -----------------------------------

def ensure_requirements_node(root):

    attributes_node = root.find(
        ".//ns1:attributes",
        namespace
    )

    if attributes_node is None:

        raise ValueError(
            "No <attributes> node found."
        )

    req_node = root.find(
        ".//ns1:requirements",
        namespace
    )

    if req_node is None:

        print(
            "\n[INFO] Creating "
            "<requirements> node."
        )

        req_node = ET.SubElement(
            attributes_node,
            f"{{{NS_PROPERTIES}}}requirements"
        )

    else:

        print(
            "\n[INFO] Existing "
            "<requirements> node found."
        )

    return req_node

# -----------------------------------
# Ensure <subscriptions> block
# -----------------------------------

def ensure_subscriptions(root):

    subscriptions = root.find(
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    if subscriptions is not None:

        print(
            "\n[INFO] Subscriptions block already exists."
        )

        return

    print(
        "\n[INFO] Creating subscriptions block."
    )

    subscriptions = ET.SubElement(
        root,
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    subscription = ET.SubElement(
        subscriptions,
        f"{{{NS_SUBSCRIPTIONS}}}subscription",
        {
            "id": "_compliance",
            "url": (
                "https://power.bpm.cit.tum.de/"
                "compliance/Subscriber"
            )
        }
    )

    topic = ET.SubElement(
        subscription,
        f"{{{NS_SUBSCRIPTIONS}}}topic",
        {
            "id": "description"
        }
    )

    event = ET.SubElement(
        topic,
        f"{{{NS_SUBSCRIPTIONS}}}event"
    )

    event.text = "change"

# -----------------------------------
# Run one PST
# -----------------------------------

def run_pst(
    scenario_name,
    pst_file
):

    print("\n===================================")
    print(f"SCENARIO: {scenario_name}")
    print(f"PST: {pst_file.name}")
    print("===================================\n")

    requirements_file = (
        BASE_DIR
        / "data"
        / "input"
        / "compliance_requirements"
        / f"{scenario_name}.json"
    )

    with open(
        requirements_file,
        "r",
        encoding="utf-8"
    ) as f:

        all_requirements = json.load(f)

    if VERIFY_ALL:

        requirements = all_requirements

        print(
            "\nMode: VERIFY ALL REQUIREMENTS"
        )

    else:

        print(
            "\nMode: VERIFY SELECTED REQUIREMENTS"
        )

        missing_requirements = [
            rid
            for rid in selected_requirement_ids
            if rid not in all_requirements
        ]

        if missing_requirements:

            raise ValueError(
                f"Requirements not found: "
                f"{missing_requirements}"
            )

        requirements = {
            rid: all_requirements[rid]
            for rid in selected_requirement_ids
        }

    requirements_text = json.dumps(
        requirements,
        ensure_ascii=False
    )

    print("\nInjected requirements:")
    print(requirements_text)

    tree = ET.parse(pst_file)

    root = tree.getroot()

    req_node = ensure_requirements_node(root)

    ensure_subscriptions(root)

    req_node.text = requirements_text

    print(
        "\n[INFO] Requirements successfully injected."
    )

    with tempfile.NamedTemporaryFile(
        suffix=".xml",
        delete=False
    ) as tmp:

        temp_xml_path = Path(tmp.name)

    tree.write(
        temp_xml_path,
        encoding="utf-8",
        xml_declaration=True
    )

    print(
        f"\nTemporary XML created:\n"
        f"{temp_xml_path}"
    )

    result = subprocess.run(
        [
            "python3",
            str(PTV_SCRIPT),
            str(temp_xml_path)
        ],
        capture_output=True,
        text=True
    )

    print("\n=== STDOUT ===")
    print(result.stdout)

    print("\n=== STDERR ===")
    print(result.stderr)

    marker = "===== VIOLATION REPORT ====="

    if marker in result.stdout:

        violation_json = (
            result.stdout
            .split(marker)[-1]
            .strip()
        )

        violation_report = json.loads(
            violation_json
        )

        # -----------------------------------
        # scenario output directory
        # -----------------------------------

        scenario_output_dir = (
            OUTPUT_DIR
            / scenario_name
        )

        scenario_output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # -----------------------------------
        # output filename
        # -----------------------------------

        output_file = (
            scenario_output_dir
            / (
                f"{pst_file.stem}"
                f"_violations.json"
            )
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                violation_report,
                f,
                indent=4,
                ensure_ascii=False
            )

        print(
            f"\nViolation report saved to:\n"
            f"{output_file}"
        )

    else:

        print(
            "\nNo violation report found."
        )

    temp_xml_path.unlink(
        missing_ok=True
    )

    print(
        "\nTemporary XML deleted."
    )

# -----------------------------------
# Main execution
# -----------------------------------

scenario_dirs = sorted(
    UPDATED_PST_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    if (
        not RUN_ALL_SCENARIOS
        and scenario_name != SELECTED_SCENARIO
    ):
        continue

    pst_dir = (
        scenario_dir
        / "pst"
    )

    if not pst_dir.exists():

        print(
            f"\nNo pst folder found for:"
            f" {scenario_name}"
        )

        continue

    pst_files = sorted(
        pst_dir.glob("*.xml")
    )

    print("\n===================================")
    print(f"FOUND {len(pst_files)} PST FILES")
    print(f"FOR SCENARIO {scenario_name}")
    print("===================================\n")

    for pst_file in pst_files:

        run_pst(
            scenario_name,
            pst_file
        )



In [ ]:
# ============================================================
# ANALYSIS OF RESOLVED AND UNRESOLVED VIOLATIONS
# ============================================================
#
# This script analyzes whether the generated resolution
# strategies successfully resolved the targeted compliance
# violations in the updated Process Structure Trees (PSTs).
#
# The script identifies:
# - resolved violations
# - unresolved violations
# - requirements never resolved
#
# Output:
# - resolution analysis summaries (.xlsx)
#
# ============================================================

# !pip install openpyxl
import json
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================


# Project root directory
BASEDIR = Path(__file__).resolve().parent.parent
VIOLATIONS_DIR = (
    BASE_DIR
    / "data/output/compliance_violations_after_changes"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data/output/resolution_strategy_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# RESULTS
# ============================================================

summary_rows = []

# ============================================================
# ITERATE SCENARIOS
# ============================================================

scenario_dirs = sorted(
    VIOLATIONS_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    print("\n===================================")
    print("SCENARIO:", scenario_name)
    print("===================================\n")

    violation_files = sorted(
        scenario_dir.glob("*.json")
    )

    for violation_file in violation_files:

        print("Processing:", violation_file.name)

        # ----------------------------------------------------
        # Extract requirement id from filename
        # ----------------------------------------------------

        # Example:
        # 01_awad_delivery_of_goods_R1_RS1_xxx_violations.json

        stem = violation_file.stem

        parts = stem.split("_")

        requirement_id = None

        for part in parts:

            if part.startswith("R"):

                if (
                    len(part) > 1
                    and part[1:].isdigit()
                ):

                    requirement_id = part
                    break

        if requirement_id is None:

            print(
                "Could not determine "
                "requirement id."
            )

            continue

        # ----------------------------------------------------
        # Load violations
        # ----------------------------------------------------

        with open(
            violation_file,
            "r",
            encoding="utf-8"
        ) as f:

            violations = json.load(f)

        # ----------------------------------------------------
        # Determine if target requirement
        # still violated
        # ----------------------------------------------------

        still_violated = False

        matching_violation = None

        for violation in violations:

            if (
                violation["requirement_id"]
                == requirement_id
            ):

                still_violated = True

                matching_violation = violation

                break

        # ----------------------------------------------------
        # Determine status
        # ----------------------------------------------------

        if still_violated:

            status = "NOT_FIXED"

            evidence = (
                matching_violation.get(
                    "evidence",
                    []
                )
            )

        else:

            status = "FIXED"

            evidence = []

        # ----------------------------------------------------
        # Add row
        # ----------------------------------------------------

        summary_rows.append({

            "scenario":
                scenario_name,

            "pst_file":
                violation_file.name,

            "requirement_id":
                requirement_id,

            "status":
                status,

            "remaining_evidence":
                " | ".join(evidence)
        })

# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(summary_rows)

# ============================================================
# SORT
# ============================================================

df = df.sort_values(
    by=[
        "scenario",
        "requirement_id",
        "pst_file"
    ]
)

# ============================================================
# REQUIREMENTS NEVER FIXED
# ============================================================

grouped = df.groupby(
    ["scenario", "requirement_id"]
)

never_fixed_rows = []

for (
    scenario,
    requirement_id
), group in grouped:

    statuses = set(
        group["status"].tolist()
    )

    # --------------------------------------------------------
    # requirement fixed at least once
    # --------------------------------------------------------

    if "FIXED" in statuses:

        overall_status = (
            "FIXED_AT_LEAST_ONCE"
        )

    # --------------------------------------------------------
    # never fixed
    # --------------------------------------------------------

    else:

        overall_status = "NEVER_FIXED"

    never_fixed_rows.append({

        "scenario":
            scenario,

        "requirement_id":
            requirement_id,

        "overall_status":
            overall_status,

        "total_strategies":
            len(group),

        "fixed_strategies":
            len(
                group[
                    group["status"]
                    == "FIXED"
                ]
            ),

        "not_fixed_strategies":
            len(
                group[
                    group["status"]
                    == "NOT_FIXED"
                ]
            )
    })

# ============================================================
# CREATE OVERVIEW DATAFRAME
# ============================================================

overview_df = pd.DataFrame(
    never_fixed_rows
)

overview_df = overview_df.sort_values(
    by=[
        "scenario",
        "requirement_id"
    ]
)



# ============================================================
# SAVE EXCEL
# ============================================================

excel_output = (
    OUTPUT_DIR
    / "resolution_strategy_summary.xlsx"
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # detailed results
    # --------------------------------------------------------

    df.to_excel(
        writer,
        sheet_name="all_results",
        index=False
    )

    # --------------------------------------------------------
    # overview
    # --------------------------------------------------------

    overview_df.to_excel(
        writer,
        sheet_name="requirements_overview",
        index=False
    )

    # --------------------------------------------------------
    # only never fixed
    # --------------------------------------------------------

    overview_df[
        overview_df["overall_status"]
        == "NEVER_FIXED"
    ].to_excel(
        writer,
        sheet_name="never_fixed",
        index=False
    )

# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n===================================")
print("SUMMARY")
print("===================================\n")

print(df)

print("\n===================================")
print("REQUIREMENTS OVERVIEW")
print("===================================\n")

print(overview_df)


print("\nExcel saved to:")
print(excel_output)

In [ ]:
# ============================================================
# ANALYSIS:
# RESOLVED VIOLATIONS BY SCENARIO
# ============================================================
#
# This script analyzes the resolution strategy summary
# and determines how many violated requirements were
# resolved for each scenario.
#
# A requirement is considered RESOLVED if at least one
# resolution strategy fixed the violation.
#
# Output:
# - CSV summary grouped by scenario
#
# ============================================================

import pandas as pd
from pathlib import Path

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# INPUT FILE
# ============================================================

INPUT_XLSX = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "resolution_strategy_summary.xlsx"
)

print("\nReading:")
print(INPUT_XLSX)

# ============================================================
# OUTPUT FILE
# ============================================================

OUTPUT_CSV = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "resolved_violations_by_scenario.csv"
)

# ============================================================
# LOAD EXCEL
# ============================================================

df = pd.read_excel(INPUT_XLSX)

print("\nLoaded rows:", len(df))

# ============================================================
# NORMALIZE STATUS COLUMN
# ============================================================

df["status"] = (
    df["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# ============================================================
# DETERMINE REQUIREMENT RESOLUTION
# ============================================================

requirement_resolution = []

grouped_requirements = df.groupby(
    ["scenario", "requirement_id"]
)

for (
    (scenario, requirement_id),
    group
) in grouped_requirements:

    resolved = (
        group["status"]
        == "FIXED"
    ).any()

    requirement_resolution.append({

        "scenario":
            scenario,

        "requirement_id":
            requirement_id,

        "resolved":
            resolved
    })

requirements_df = pd.DataFrame(
    requirement_resolution
)

# ============================================================
# GROUP BY SCENARIO
# ============================================================

scenario_summary = []

grouped_scenarios = requirements_df.groupby(
    "scenario"
)

for scenario, group in grouped_scenarios:

    total_requirements = len(group)

    resolved_requirements = (
        group["resolved"]
        == True
    ).sum()

    unresolved_requirements = (
        group["resolved"]
        == False
    ).sum()

    resolution_rate = round(
        (
            resolved_requirements
            / total_requirements
        ) * 100,
        2
    )

    scenario_summary.append({

        "scenario":
            scenario,

        "total_violated_requirements":
            total_requirements,

        "resolved_requirements":
            resolved_requirements,

        "unresolved_requirements":
            unresolved_requirements,

        "resolution_rate_percent":
            resolution_rate
    })

# ============================================================
# CREATE SUMMARY DATAFRAME
# ============================================================

summary_df = pd.DataFrame(
    scenario_summary
)

summary_df = summary_df.sort_values(
    by="scenario"
)

# ============================================================
# SAVE CSV
# ============================================================

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("RESOLVED VIOLATIONS BY SCENARIO")
print("================================================")

print(summary_df)

print("\nSaved CSV:")
print(OUTPUT_CSV)

In [ ]:
# ============================================================
# ANALYSIS:
# CHANGE OPERATIONS BY SCENARIO
# ============================================================
#
# This script analyzes the generated resolution strategies
# and counts how many times each change operation was used
# per scenario.
#
# Example:
# - copy_after -> 4
# - insert_before -> 2
#
# Output:
# - CSV summary of change operation frequencies
#   grouped by scenario
#
# ============================================================

import json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# INPUT DIRECTORY
# ============================================================

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "generated_resolution_strategies"
)

print("\nReading resolution strategies from:")
print(INPUT_DIR)

# ============================================================
# OUTPUT FILE
# ============================================================

OUTPUT_CSV = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "change_operations_by_scenario.csv"
)

# ============================================================
# INITIALIZATION
# ============================================================

scenario_operation_counter = defaultdict(
    Counter
)

# ============================================================
# ITERATE OVER SCENARIOS
# ============================================================

scenario_dirs = [

    d for d in INPUT_DIR.iterdir()

    if d.is_dir()
]

for scenario_dir in scenario_dirs:

    scenario_name = scenario_dir.name

    resolution_dir = (
        scenario_dir
        / "resolution_strategies_clean"
    )

    if not resolution_dir.exists():

        continue

    print("\nProcessing scenario:")
    print(scenario_name)

    json_files = list(
        resolution_dir.glob("*.json")
    )

    for json_file in json_files:

        try:

            with open(
                json_file,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"Failed reading "
                f"{json_file.name}: {e}"
            )

            continue

        resolution_strategies = data.get(
            "resolution_strategies",
            []
        )

        for strategy in resolution_strategies:

            change_operations = strategy.get(
                "change_operations",
                []
            )

            for operation in change_operations:

                operation_name = operation.get(
                    "operation"
                )

                if operation_name:

                    scenario_operation_counter[
                        scenario_name
                    ][operation_name] += 1

# ============================================================
# CREATE SUMMARY ROWS
# ============================================================

summary_rows = []

for (
    scenario,
    counter
) in scenario_operation_counter.items():

    total_operations = sum(
        counter.values()
    )

    for (
        operation,
        count
    ) in counter.items():

        percentage = round(
            (
                count
                / total_operations
            ) * 100,
            2
        )

        summary_rows.append({

            "scenario":
                scenario,

            "operation":
                operation,

            "count":
                count,

            "percentage":
                percentage
        })

# ============================================================
# CREATE DATAFRAME
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_df = summary_df.sort_values(
    by=[
        "scenario",
        "count"
    ],
    ascending=[
        True,
        False
    ]
)

# ============================================================
# SAVE CSV
# ============================================================

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("CHANGE OPERATIONS BY SCENARIO")
print("================================================")

print(summary_df)

print("\nSaved CSV:")
print(OUTPUT_CSV)

In [ ]:
# ============================================================
# ANALYSIS:
# CHANGE OPERATIONS THAT FIXED VIOLATIONS
# VS
# CHANGE OPERATIONS THAT DID NOT
# ============================================================
#
# This script combines:
#
# 1. Generated resolution strategies
# 2. Resolution strategy execution results
#
# and classifies:
#
# - Which change operations contributed to FIXED violations
# - Which change operations belonged to NOT_FIXED violations
#
# Output:
# - CSV with global operation statistics
#
# ============================================================

import json
import pandas as pd
from pathlib import Path
from collections import Counter

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# INPUTS
# ============================================================

STRATEGY_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "generated_resolution_strategies"
)

SUMMARY_XLSX = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "resolution_strategy_summary.xlsx"
)

print("\nReading strategy results:")
print(SUMMARY_XLSX)

print("\nReading strategies from:")
print(STRATEGY_DIR)

# ============================================================
# OUTPUT FILE
# ============================================================

OUTPUT_CSV = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "change_operations_fix_effectiveness.csv"
)

# ============================================================
# LOAD SUMMARY EXCEL
# ============================================================

summary_df = pd.read_excel(
    SUMMARY_XLSX
)

summary_df["status"] = (
    summary_df["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# ============================================================
# COUNTERS
# ============================================================

fixed_counter = Counter()

not_fixed_counter = Counter()

total_counter = Counter()

# ============================================================
# ITERATE OVER RESULT ROWS
# ============================================================

for _, row in summary_df.iterrows():

    scenario = row["scenario"]

    pst_file = row["pst_file"]

    status = row["status"]

    # --------------------------------------------------------
    # EXTRACT STRATEGY ID
    # --------------------------------------------------------

    # Example:
    # 01_awad_delivery_of_goods_R1_RS1_copy_notification...
    #
    # -> RS1_copy_notification...

    split_name = pst_file.split("_")

    rs_index = None

    for i, token in enumerate(split_name):

        if token.startswith("RS"):

            rs_index = i
            break

    if rs_index is None:

        continue

    strategy_id = "_".join(
        split_name[rs_index:]
    )

    strategy_id = strategy_id.replace(
        "_violations.json",
        ""
    )

    # --------------------------------------------------------
    # FIND STRATEGY JSON FILES
    # --------------------------------------------------------

    strategy_folder = (
        STRATEGY_DIR
        / scenario
        / "resolution_strategies_clean"
    )

    if not strategy_folder.exists():

        continue

    json_files = list(
        strategy_folder.glob("*.json")
    )

    matched_strategy = None

    # --------------------------------------------------------
    # SEARCH STRATEGY
    # --------------------------------------------------------

    for json_file in json_files:

        try:

            with open(
                json_file,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"Failed reading "
                f"{json_file.name}: {e}"
            )

            continue

        strategies = data.get(
            "resolution_strategies",
            []
        )

        for strategy in strategies:

            if (
                strategy.get(
                    "resolution_strategy_id"
                )
                == strategy_id
            ):

                matched_strategy = strategy
                break

        if matched_strategy:

            break

    # --------------------------------------------------------
    # NO MATCH FOUND
    # --------------------------------------------------------

    if matched_strategy is None:

        print(
            f"Strategy not found: "
            f"{strategy_id}"
        )

        continue

    # --------------------------------------------------------
    # COUNT OPERATIONS
    # --------------------------------------------------------

    change_operations = matched_strategy.get(
        "change_operations",
        []
    )

    for operation in change_operations:

        operation_name = operation.get(
            "operation"
        )

        if not operation_name:

            continue

        total_counter[
            operation_name
        ] += 1

        if status == "FIXED":

            fixed_counter[
                operation_name
            ] += 1

        else:

            not_fixed_counter[
                operation_name
            ] += 1

# ============================================================
# CREATE SUMMARY TABLE
# ============================================================

summary_rows = []

all_operations = sorted(
    total_counter.keys()
)

for operation in all_operations:

    total = total_counter[operation]

    fixed = fixed_counter[operation]

    not_fixed = not_fixed_counter[operation]

    fix_rate = round(
        (
            fixed / total
        ) * 100,
        2
    ) if total > 0 else 0

    summary_rows.append({

        "operation":
            operation,

        "total_usage":
            total,

        "fixed_usage":
            fixed,

        "not_fixed_usage":
            not_fixed,

        "fix_rate_percent":
            fix_rate
    })

# ============================================================
# CREATE DATAFRAME
# ============================================================

result_df = pd.DataFrame(
    summary_rows
)

result_df = result_df.sort_values(
    by="fix_rate_percent",
    ascending=False
)

# ============================================================
# SAVE CSV
# ============================================================

result_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("CHANGE OPERATIONS FIX EFFECTIVENESS")
print("================================================")

print(result_df)

print("\nSaved CSV:")
print(OUTPUT_CSV)

In [ ]:
# ============================================================
# ANALYSIS:
# CHANGE COSTS BY SCENARIO
# ============================================================
#
# This script calculates redesign/change costs
# per scenario based on the generated resolution
# strategies.
#
# The cost model reflects the structural and
# behavioral impact of each change operation.
#
# Costs are normalized by the number of
# resolved violations.
#
# Output:
# - CSV with normalized redesign costs
#   per scenario
#
# ============================================================

import json
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# INPUT DIRECTORIES
# ============================================================

STRATEGY_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "generated_resolution_strategies"
)

RESOLVED_VIOLATIONS_CSV = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "resolved_violations_by_scenario.csv"
)

print("\nReading resolution strategies from:")
print(STRATEGY_DIR)

print("\nReading resolved violations from:")
print(RESOLVED_VIOLATIONS_CSV)

# ============================================================
# OUTPUT FILE
# ============================================================

OUTPUT_CSV = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
    / "change_costs_by_scenario.csv"
)

# ============================================================
# CHANGE COST MODEL
# ============================================================

CHANGE_COSTS = {

    # --------------------------------------------------------
    # LOW IMPACT
    # --------------------------------------------------------

    "rename": 1,
    "modify_resource": 1,
    "modify_write": 1,
    "add_write": 1,
    "remove_write": 1,
    "modify_read": 1,
    "modify_condition": 1,
    "modify_loop_condition": 1,
    "modify_timeout": 1,

    # --------------------------------------------------------
    # MODERATE IMPACT
    # --------------------------------------------------------

    "insert_after": 2,
    "insert_before": 2,
    "delete": 2,
    "move_after": 2,
    "move_before": 2,
    "copy_after": 2,
    "copy_before": 2,
    "swap": 2,
    "merge": 2,
    "split": 2,

    # --------------------------------------------------------
    # HIGH IMPACT
    # --------------------------------------------------------

    "parallelize": 3,
    "sequentialize_parallel": 3,
    "add_xor_branch": 3,
    "remove_branch": 3,
    "remove_branch_by_condition": 3,
    "embed_activity_in_xor": 3,
    "remove_loop": 3,

    # --------------------------------------------------------
    # VERY HIGH IMPACT
    # --------------------------------------------------------

    "embed_pre_loop": 4,
    "embed_post_loop": 4
}

# ============================================================
# LOAD RESOLVED VIOLATIONS
# ============================================================

resolved_df = pd.read_csv(
    RESOLVED_VIOLATIONS_CSV
)

resolved_lookup = {}

for _, row in resolved_df.iterrows():

    scenario = row["scenario"]

    resolved_lookup[scenario] = int(
        row["resolved_requirements"]
    )

# ============================================================
# INITIALIZATION
# ============================================================

scenario_statistics = defaultdict(
    lambda: {

        "total_cost": 0,

        "total_operations": 0,

        "total_strategies": 0,

        "strategy_costs": []
    }
)

# ============================================================
# ITERATE OVER SCENARIOS
# ============================================================

scenario_dirs = [

    d for d in STRATEGY_DIR.iterdir()

    if d.is_dir()
]

for scenario_dir in scenario_dirs:

    scenario_name = scenario_dir.name

    resolution_dir = (
        scenario_dir
        / "resolution_strategies_clean"
    )

    if not resolution_dir.exists():

        continue

    print("\nProcessing scenario:")
    print(scenario_name)

    json_files = list(
        resolution_dir.glob("*.json")
    )

    for json_file in json_files:

        try:

            with open(
                json_file,
                "r",
                encoding="utf-8"
            ) as f:

                data = json.load(f)

        except Exception as e:

            print(
                f"Failed reading "
                f"{json_file.name}: {e}"
            )

            continue

        resolution_strategies = data.get(
            "resolution_strategies",
            []
        )

        # ----------------------------------------------------
        # PROCESS EACH STRATEGY
        # ----------------------------------------------------

        for strategy in resolution_strategies:

            strategy_cost = 0

            change_operations = strategy.get(
                "change_operations",
                []
            )

            for operation in change_operations:

                operation_name = operation.get(
                    "operation"
                )

                if not operation_name:

                    continue

                operation_cost = CHANGE_COSTS.get(
                    operation_name,
                    1
                )

                strategy_cost += operation_cost

                scenario_statistics[
                    scenario_name
                ]["total_cost"] += operation_cost

                scenario_statistics[
                    scenario_name
                ]["total_operations"] += 1

            scenario_statistics[
                scenario_name
            ]["total_strategies"] += 1

            scenario_statistics[
                scenario_name
            ]["strategy_costs"].append(
                strategy_cost
            )

# ============================================================
# CREATE SUMMARY TABLE
# ============================================================

summary_rows = []

for (
    scenario,
    stats
) in scenario_statistics.items():

    total_cost = stats["total_cost"]

    total_operations = stats[
        "total_operations"
    ]

    total_strategies = stats[
        "total_strategies"
    ]

    strategy_costs = stats[
        "strategy_costs"
    ]

    resolved_violations = resolved_lookup.get(
        scenario,
        0
    )

    avg_cost_per_strategy = round(
        (
            total_cost
            / total_strategies
        ),
        2
    ) if total_strategies > 0 else 0

    avg_cost_per_operation = round(
        (
            total_cost
            / total_operations
        ),
        2
    ) if total_operations > 0 else 0

    cost_per_resolved_violation = round(
        (
            total_cost
            / resolved_violations
        ),
        2
    ) if resolved_violations > 0 else 0

    max_strategy_cost = max(
        strategy_costs
    ) if strategy_costs else 0

    min_strategy_cost = min(
        strategy_costs
    ) if strategy_costs else 0

    summary_rows.append({

        "scenario":
            scenario,

        "resolved_violations":
            resolved_violations,

        "total_change_cost":
            total_cost,

        "cost_per_resolved_violation":
            cost_per_resolved_violation,

        "total_resolution_strategies":
            total_strategies,

        "total_operations":
            total_operations,

        "avg_cost_per_strategy":
            avg_cost_per_strategy,

        "avg_cost_per_operation":
            avg_cost_per_operation,

        "max_strategy_cost":
            max_strategy_cost,

        "min_strategy_cost":
            min_strategy_cost
    })

# ============================================================
# CREATE DATAFRAME
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_df = summary_df.sort_values(
    by="cost_per_resolved_violation",
    ascending=False
)

# ============================================================
# SAVE CSV
# ============================================================

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("CHANGE COSTS BY SCENARIO")
print("================================================")

print(summary_df)

print("\nSaved CSV:")
print(OUTPUT_CSV)

In [ ]:
# Analysis overlap between requirements presenting violations and those which not (resolution context)


import os
import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# Project root
BASE_PATH = Path(__file__).resolve().parents[2]

# Requirements WITH violations
VIOLATION_REQS_DIR = (
    BASE_PATH / "data" / "input" / "compliance_requirements"
)

# Resolution context requirements
RESOLUTION_CONTEXT_DIR = (
    BASE_PATH / "data" / "input" / "resolution_context"
)

# Output directory
OUTPUT_DIR = (
    BASE_PATH / "data" / "output" / "analysis_resolution_context"
)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# REGEX
# ============================================================

PATTERN_REGEX = re.compile(r'(\w+)\((.*?)\)')


# ============================================================
# HELPERS
# ============================================================

def split_arguments(arg_string):

    args = re.findall(r"'([^']*)'|([^,]+)", arg_string)

    cleaned = []

    for a, b in args:
        val = a if a else b
        cleaned.append(val.strip())

    return cleaned


def extract_elements(requirement_text):

    result = {
        "activities": set(),
        "resources": set(),
        "data": set(),
        "temporal": set(),
    }

    matches = PATTERN_REGEX.findall(requirement_text)

    for pattern_name, arg_string in matches:

        args = split_arguments(arg_string)

        # ----------------------------------------------------
        # ACTIVITY ONLY
        # ----------------------------------------------------
        if pattern_name in {
            "exists",
            "absence",
            "loop",
        }:
            if len(args) >= 1:
                result["activities"].add(args[0])

        # ----------------------------------------------------
        # TEMPORAL
        # ----------------------------------------------------
        elif pattern_name in {
            "directly_follows",
            "leads_to",
            "precedence",
            "leads_to_absence",
            "precedence_absence",
            "parallel",
            "condition_directly_follows",
            "condition_eventually_follows",
        }:
            if len(args) >= 2:
                result["activities"].update([args[0], args[1]])
                result["temporal"].add(pattern_name)

        # ----------------------------------------------------
        # RESOURCE
        # ----------------------------------------------------
        elif pattern_name == "executed_by":

            if len(args) >= 2:
                result["activities"].add(args[0])
                result["resources"].add(args[1])

        # ----------------------------------------------------
        # TIMED
        # ----------------------------------------------------
        elif pattern_name == "timed_alternative":

            if len(args) >= 3:
                result["activities"].update([args[0], args[1]])
                result["temporal"].add("timed_alternative")
                result["temporal"].add(args[2])

        # ----------------------------------------------------
        # DATA
        # ----------------------------------------------------
        elif pattern_name in {
            "activity_sends",
            "activity_receives",
            "data_leads_to_absence",
        }:

            if len(args) >= 2:
                result["activities"].add(args[0])
                result["data"].add(args[1])

    return result


def load_json(filepath):

    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def get_scenario_name(filename):

    filename = filename.replace(".json", "")
    filename = filename.replace("_req_resolution_context", "")

    return filename


# ============================================================
# LOAD FILES
# ============================================================

violation_files = {
    get_scenario_name(f): os.path.join(VIOLATION_REQS_DIR, f)
    for f in os.listdir(VIOLATION_REQS_DIR)
    if f.endswith(".json")
}

resolution_files = {
    get_scenario_name(f): os.path.join(RESOLUTION_CONTEXT_DIR, f)
    for f in os.listdir(RESOLUTION_CONTEXT_DIR)
    if f.endswith(".json")
}

# ============================================================
# MATCH SCENARIOS
# ============================================================

common_scenarios = sorted(
    set(violation_files.keys())
    & set(resolution_files.keys())
)

# ============================================================
# ANALYSIS
# ============================================================

summary_rows = []

for scenario in common_scenarios:

    print(f"Processing {scenario}")

    violation_path = violation_files[scenario]
    resolution_path = resolution_files[scenario]

    violation_reqs = load_json(violation_path)
    resolution_reqs = load_json(resolution_path)

    violation_elements = defaultdict(set)
    resolution_elements = defaultdict(set)

    # --------------------------------------------------------
    # Extract violation requirements
    # --------------------------------------------------------
    for rid, req_text in violation_reqs.items():

        extracted = extract_elements(req_text)

        for k, v in extracted.items():
            violation_elements[k].update(v)

    # --------------------------------------------------------
    # Extract resolution context requirements
    # --------------------------------------------------------
    for rid, req_text in resolution_reqs.items():

        extracted = extract_elements(req_text)

        for k, v in extracted.items():
            resolution_elements[k].update(v)

    # --------------------------------------------------------
    # Compute overlaps
    # --------------------------------------------------------
    activity_overlap = sorted(
        violation_elements["activities"]
        & resolution_elements["activities"]
    )

    resource_overlap = sorted(
        violation_elements["resources"]
        & resolution_elements["resources"]
    )

    data_overlap = sorted(
        violation_elements["data"]
        & resolution_elements["data"]
    )

    temporal_overlap = sorted(
        violation_elements["temporal"]
        & resolution_elements["temporal"]
    )

    # --------------------------------------------------------
    # Save JSON result
    # --------------------------------------------------------
    per_scenario_result = {
        "activities": activity_overlap,
        "resources": resource_overlap,
        "data": data_overlap,
        "temporal": temporal_overlap,
    }

    json_output = os.path.join(
        OUTPUT_DIR,
        f"{scenario}_overlap_analysis.json"
    )

    with open(json_output, "w", encoding="utf-8") as f:
        json.dump(per_scenario_result, f, indent=4)

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------
    summary_rows.append({
        "scenario": scenario,

        "activity_overlap_count": len(activity_overlap),
        "resource_overlap_count": len(resource_overlap),
        "data_overlap_count": len(data_overlap),
        "temporal_overlap_count": len(temporal_overlap),

        "activity_overlaps": ", ".join(activity_overlap),
        "resource_overlaps": ", ".join(resource_overlap),
        "data_overlaps": ", ".join(data_overlap),
        "temporal_overlaps": ", ".join(temporal_overlap),
    })

# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(summary_rows)

# ============================================================
# SAVE EXCEL
# ============================================================

excel_output = os.path.join(
    OUTPUT_DIR,
    "summary_overlap_analysis.xlsx"
)

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Overlap Summary",
        index=False
    )

# ============================================================
# FORMAT EXCEL
# ============================================================

wb = load_workbook(excel_output)
ws = wb["Overlap Summary"]

# Bold headers
for cell in ws[1]:
    cell.font = Font(bold=True)

# Auto column width
for column_cells in ws.columns:

    length = max(
        len(str(cell.value)) if cell.value else 0
        for cell in column_cells
    )

    ws.column_dimensions[
        column_cells[0].column_letter
    ].width = min(length + 5, 60)

wb.save(excel_output)

# ============================================================
# FINISHED
# ============================================================

print("\n================================================")
print("ANALYSIS FINISHED")
print("================================================")

print(f"\nExcel summary saved to:\n{excel_output}")

In [ ]:
# 2nd version of Step 1, considering the resolution context derived from the requirements that were initially compliant
# ============================================================
# STEP 1 -  V2
# GENERATE RESOLUTION STRATEGIES
# ============================================================
#
# This script generates candidate resolution strategies for
# detected compliance violations using a Large Language Model
# (LLM).
#
# For a selected scenario and violated requirement, the script:
# - loads the simplified Process Structure Tree (PST)
# - loads the detected compliance violation
# - loads resolution context requirements
# - constructs a prompting context
# - queries the LLM for candidate repair strategies
# - stores the generated strategies and metadata
#
# The generated resolution strategies consist of sequences of
# change operations intended to mitigate or resolve the
# detected compliance violation while preserving already
# compliant requirements whenever possible.
#
# Output:
# - generated resolution strategies (.json)
# - raw LLM responses
# - metadata and execution statistics
#
# ============================================================

import json
import time
import requests
from pathlib import Path

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()
BASE_DIR = CURRENT_DIR

TARGET = "PTResolver"

while BASE_DIR.name != TARGET:
    # stop if we reached filesystem root
    if BASE_DIR.parent == BASE_DIR:
        raise FileNotFoundError(
            f"Could not find '{TARGET}' in parent directories."
        )

    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# CONFIGURATION
# ============================================================

SCENARIO_NAME = "02_de_masellis_loan_approval"

REQUIREMENT_ID = "R1"

MODEL = "openai/gpt-5.5"

# ============================================================
# LOAD API KEY
# ============================================================

API_KEY_FILE = (
    BASE_DIR
    / "config"
    / "api_keys.json"
)

with open(
    API_KEY_FILE,
    "r",
    encoding="utf-8"
) as f:

    api_config = json.load(f)

OPENROUTER_API_KEY = (
    api_config["OPENROUTER_API_KEY"]
)

# ============================================================
# INPUT FILES
# ============================================================

PROMPT_FILE = (
    BASE_DIR
    / "data"
    / "input"
    / "prompts"
    / "V1_generate_resolution_strategies_no_process_change_operations_free_answer.txt"
)

PST_FILE = (
    BASE_DIR
    / "data"
    / "output"
    / "simplified_pst"
    / f"{SCENARIO_NAME}_simplified_pst.txt"
)

VIOLATIONS_FILE = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_before_changes"
    / f"{SCENARIO_NAME}_ALL_violations.json"
)

# ============================================================
# RESOLUTION CONTEXT FILE
# ============================================================

RESOLUTION_CONTEXT_FILE = (
    BASE_DIR
    / "data"
    / "input"
    / "resolution_context"
    / f"{SCENARIO_NAME}_req_resolution_context.json"
)

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "generated_resolution_strategies"
)

SCENARIO_OUTPUT_DIR = (
    OUTPUT_DIR
    / SCENARIO_NAME
)

JSON_DIR = (
    SCENARIO_OUTPUT_DIR
    / "resolution_strategies_clean"
)

RAW_DIR = (
    SCENARIO_OUTPUT_DIR
    / "raw"
)

FULL_RESPONSE_DIR = (
    SCENARIO_OUTPUT_DIR
    / "full_response"
)

METADATA_DIR = (
    SCENARIO_OUTPUT_DIR
    / "metadata"
)

PROBLEMS_DIR = (
    SCENARIO_OUTPUT_DIR
    / "problems"
)

JSON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FULL_RESPONSE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROBLEMS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# TIMER START
# ============================================================

start_time = time.time()

# ============================================================
# LOGGING VARIABLES
# ============================================================

problems_detected = []

# ============================================================
# LOAD BASE PROMPT
# ============================================================

with open(
    PROMPT_FILE,
    "r",
    encoding="utf-8"
) as f:

    BASE_PROMPT = f.read()

# ============================================================
# LOAD PST
# ============================================================

with open(
    PST_FILE,
    "r",
    encoding="utf-8"
) as f:

    pst = f.read()

# ============================================================
# LOAD VIOLATIONS
# ============================================================

with open(
    VIOLATIONS_FILE,
    "r",
    encoding="utf-8"
) as f:

    violations = json.load(f)

# ============================================================
# LOAD RESOLUTION CONTEXT
# ============================================================

resolution_context = {}

if RESOLUTION_CONTEXT_FILE.exists():

    with open(
        RESOLUTION_CONTEXT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        resolution_context = json.load(f)

    print(
        f"Loaded resolution context requirements: "
        f"{len(resolution_context)}"
    )

else:

    print(
        "No resolution context file found."
    )

# ============================================================
# FIND TARGET VIOLATION
# ============================================================

target_violation = None

for violation in violations:

    if violation["requirement_id"] == REQUIREMENT_ID:

        target_violation = violation

        break

if target_violation is None:

    raise ValueError(
        f"Violation '{REQUIREMENT_ID}' not found."
    )

print(
    f"Using violation: "
    f"{REQUIREMENT_ID}"
)

# ============================================================
# BUILD OUTPUT PREFIX
# ============================================================

OUTPUT_PREFIX = (
    f"{SCENARIO_NAME}_RS_{REQUIREMENT_ID}"
)

print(
    "Output prefix:",
    OUTPUT_PREFIX
)

# ============================================================
# BUILD PROMPT
# ============================================================

final_prompt = f"""{BASE_PROMPT}

============================================================
PROCESS STRUCTURED TREE
============================================================

{pst}

============================================================
DETECTED VIOLATION
============================================================

{json.dumps(target_violation, indent=2)}

============================================================
RESOLUTION CONTEXT REQUIREMENTS
============================================================

These requirements are currently satisfied and should remain
satisfied whenever possible.

Avoid introducing new violations of these requirements.

{json.dumps(resolution_context, indent=2)}

IMPORTANT:
- Return ONLY valid JSON
- Do not use markdown
- Do not use code fences
- Ensure output is parseable with json.loads()
"""

# ============================================================
# API REQUEST
# ============================================================

print(
    "\nSending request to model...\n"
)

try:

    response = requests.post(
        url=(
            "https://openrouter.ai/"
            "api/v1/chat/completions"
        ),
        headers={
            "Authorization": (
                f"Bearer "
                f"{OPENROUTER_API_KEY}"
            ),
            "Content-Type":
                "application/json",
        },
        data=json.dumps({
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": final_prompt
                }
            ],
            "temperature": 0,
            "reasoning": {
                "enabled": True
            }
        }),
        timeout=300
    )

except Exception as e:

    problems_detected.append(
        f"Request exception: {str(e)}"
    )

    raise e

# ============================================================
# CHECK RESPONSE
# ============================================================

if response.status_code != 200:

    problems_detected.append(
        f"HTTP {response.status_code}"
    )

    print("ERROR:")

    print(response.status_code)

    print(response.text)

    raise Exception(
        "API request failed"
    )

response_json = response.json()

# ============================================================
# EXTRACT MODEL OUTPUT
# ============================================================

print(
    "\n========== FULL RESPONSE ==========\n"
)

print(
    json.dumps(
        response_json,
        indent=2
    )
)

if "choices" not in response_json:

    problems_detected.append(
        "API response missing 'choices'"
    )

    raise ValueError(
        "API response does not contain "
        "'choices'. See printed response above."
    )

try:

    assistant_message = (
        response_json["choices"][0]["message"]
    )

    generated_text = (
        assistant_message["content"]
    )

except Exception as e:

    problems_detected.append(
        "Failed to extract assistant message"
    )

    raise e

# ============================================================
# VALIDATE JSON
# ============================================================

parsed_json = None

try:

    parsed_json = json.loads(
        generated_text
    )

except Exception as e:

    problems_detected.append(
        "Model output is not valid JSON"
    )

    print(
        "\nERROR: Invalid JSON generated"
    )

    print(e)

# ============================================================
# SAVE RAW RESPONSE
# ============================================================

raw_output_path = (
    RAW_DIR
    / f"{OUTPUT_PREFIX}_raw.txt"
)

with open(
    raw_output_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(generated_text)

print(
    f"\nRaw response saved to:\n"
    f"{raw_output_path}"
)

# ============================================================
# SAVE PARSED JSON
# ============================================================

if parsed_json is not None:

    json_output_path = (
        JSON_DIR
        / f"{OUTPUT_PREFIX}.json"
    )

    with open(
        json_output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed_json,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nJSON saved to:\n"
        f"{json_output_path}"
    )

# ============================================================
# SAVE FULL API RESPONSE
# ============================================================

full_response_path = (
    FULL_RESPONSE_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_full_response.json"
    )
)

with open(
    full_response_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        response_json,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nFull API response saved to:\n"
    f"{full_response_path}"
)

# ============================================================
# EXECUTION TIME
# ============================================================

end_time = time.time()

execution_time_ms = round(
    (end_time - start_time) * 1000,
    2
)

# ============================================================
# TOKEN USAGE
# ============================================================

usage = response_json.get(
    "usage",
    {}
)

# ============================================================
# SAVE METADATA
# ============================================================

metadata = {

    "model":
        MODEL,

    "scenario_name":
        SCENARIO_NAME,

    "requirement_id":
        REQUIREMENT_ID,

    "output_prefix":
        OUTPUT_PREFIX,

    "execution_time_milliseconds":
        execution_time_ms,

    "resolution_context_requirements":
        len(resolution_context),

    "problems_detected":
        problems_detected,

    "usage": {

        "prompt_tokens":
            usage.get(
                "prompt_tokens"
            ),

        "completion_tokens":
            usage.get(
                "completion_tokens"
            ),

        "total_tokens":
            usage.get(
                "total_tokens"
            ),

        "reasoning_tokens":
            usage.get(
                "reasoning_tokens"
            )
    }
}

metadata_path = (
    METADATA_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_metadata.json"
    )
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"\nMetadata saved to:\n"
    f"{metadata_path}"
)

# ============================================================
# SAVE PROBLEM LOG
# ============================================================

problem_log_path = (
    PROBLEMS_DIR
    / (
        f"{OUTPUT_PREFIX}"
        f"_problems.log"
    )
)

with open(
    problem_log_path,
    "w",
    encoding="utf-8"
) as f:

    if len(problems_detected) == 0:

        f.write(
            "No problems detected.\n"
        )

    else:

        for problem in problems_detected:

            f.write(problem + "\n")

print(
    f"\nProblem log saved to:\n"
    f"{problem_log_path}"
)

# ============================================================
# SUMMARY
# ============================================================

print(
    "\n========== EXECUTION SUMMARY ==========\n"
)

print(
    "Execution time (milliseconds):",
    execution_time_ms
)

print(
    "Prompt tokens:",
    usage.get("prompt_tokens")
)

print(
    "Completion tokens:",
    usage.get("completion_tokens")
)

print(
    "Total tokens:",
    usage.get("total_tokens")
)

print(
    "Reasoning tokens:",
    usage.get("reasoning_tokens")
)

print(
    "Resolution context requirements:",
    len(resolution_context)
)

print("\nProblems detected:")

if len(problems_detected) == 0:

    print("None")

else:

    for problem in problems_detected:

        print("-", problem)

In [ ]:
# Iterations readyyyy -step 1

import json
import time
import requests
from pathlib import Path

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()
BASE_DIR = CURRENT_DIR

TARGET = "PTResolver"

while BASE_DIR.name != TARGET:

    if BASE_DIR.parent == BASE_DIR:

        raise FileNotFoundError(
            f"Could not find '{TARGET}' in parent directories."
        )

    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# SCENARIOS + REQUIREMENTS
# ============================================================

SCENARIOS = {

    "01_awad_delivery_of_goods":
      ["R1", "R2", "R3", "R4"],

    "02_de_masellis_loan_approval":
       ["R1", "R2", "R3"],

    "03_elgammal_loan_approval":
       ["R2", "R3"],

    "04_ghose_package_screening":
        ["R1"],

   "05_loebbecke_blood_collection":
        ["R6"],

    "06_BPMQ":
        ["R2", "R4", "R5", "R6"],

     "07_CRL":
        ["R1", "R3", "R4", "R6"],

     "08_DCR":
        ["R2"],

    "09_HaarmannetAL2021":
        ["R1"],

    "11_PCL":
        ["R1", "R3", "R4"],

    "12_PPSL":
        ["R2", "R3", "R4"],

    "13_Status":
        ["R1", "R2"],

    "14_SunetAl24":
        ["R2", "R3", "R4"],

    "15_WinteretAlER":
        ["R2", "R3"],

    "16_Zasadaetal23":
        ["R2", "R3", "R5"]
}

MODEL = "openai/gpt-5.5"

# ============================================================
# LOAD API KEY
# ============================================================

API_KEY_FILE = (
    BASE_DIR
    / "config"
    / "api_keys.json"
)

with open(
    API_KEY_FILE,
    "r",
    encoding="utf-8"
) as f:

    api_config = json.load(f)

OPENROUTER_API_KEY = (
    api_config["OPENROUTER_API_KEY"]
)

# ============================================================
# LOAD BASE PROMPT ONCE
# ============================================================

PROMPT_FILE = (
    BASE_DIR
    / "data"
    / "input"
    / "prompts"
    / "final_generate_resolution_strategies.txt"
)

with open(
    PROMPT_FILE,
    "r",
    encoding="utf-8"
) as f:

    BASE_PROMPT = f.read()

# ============================================================
# MAIN EXECUTION FUNCTION
# ============================================================

def run_generation(
    scenario_name,
    requirement_id
):

    print("\n================================================")
    print(
        f"RUNNING: {scenario_name} | {requirement_id}"
    )
    print("================================================\n")

    start_time = time.time()

    problems_detected = []

    # ========================================================
    # INPUT FILES
    # ========================================================

    PST_FILE = (
        BASE_DIR
        / "data"
        / "output"
        / "simplified_pst"
        / f"{scenario_name}_simplified_pst.txt"
    )

    VIOLATIONS_FILE = (
        BASE_DIR
        / "data"
        / "output"
        / "compliance_violations_before_changes"
        / f"{scenario_name}_ALL_violations.json"
    )

    RESOLUTION_CONTEXT_FILE = (
        BASE_DIR
        / "data"
        / "input"
        / "resolution_context"
        / f"{scenario_name}_req_resolution_context.json"
    )

    # ========================================================
    # OUTPUT DIRECTORIES
    # ========================================================

    OUTPUT_DIR = (
        BASE_DIR
        / "data"
        / "output"
        / "generated_resolution_strategies"
    )

    SCENARIO_OUTPUT_DIR = (
        OUTPUT_DIR
        / scenario_name
    )

    JSON_DIR = (
        SCENARIO_OUTPUT_DIR
        / "resolution_strategies_clean"
    )

    RAW_DIR = (
        SCENARIO_OUTPUT_DIR
        / "raw"
    )

    FULL_RESPONSE_DIR = (
        SCENARIO_OUTPUT_DIR
        / "full_response"
    )

    METADATA_DIR = (
        SCENARIO_OUTPUT_DIR
        / "metadata"
    )

    PROBLEMS_DIR = (
        SCENARIO_OUTPUT_DIR
        / "problems"
    )

    for directory in [
        JSON_DIR,
        RAW_DIR,
        FULL_RESPONSE_DIR,
        METADATA_DIR,
        PROBLEMS_DIR
    ]:

        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    # ========================================================
    # LOAD PST
    # ========================================================

    with open(
        PST_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        pst = f.read()

    # ========================================================
    # LOAD VIOLATIONS
    # ========================================================

    with open(
        VIOLATIONS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        violations = json.load(f)

    # ========================================================
    # LOAD RESOLUTION CONTEXT
    # ========================================================

    resolution_context = {}

    if RESOLUTION_CONTEXT_FILE.exists():

        with open(
            RESOLUTION_CONTEXT_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            resolution_context = json.load(f)

        print(
            f"Loaded resolution context requirements: "
            f"{len(resolution_context)}"
        )

    else:

        print(
            "No resolution context file found."
        )

    # ========================================================
    # FIND TARGET VIOLATION
    # ========================================================

    target_violation = None

    for violation in violations:

        if violation["requirement_id"] == requirement_id:

            target_violation = violation
            break

    if target_violation is None:

        raise ValueError(
            f"Violation '{requirement_id}' not found."
        )

    print(
        f"Using violation: "
        f"{requirement_id}"
    )

    # ========================================================
    # BUILD OUTPUT PREFIX
    # ========================================================

    OUTPUT_PREFIX = (
        f"{scenario_name}_RS_{requirement_id}"
    )

    print(
        "Output prefix:",
        OUTPUT_PREFIX
    )

    # ========================================================
    # BUILD PROMPT
    # ========================================================

    final_prompt = f"""{BASE_PROMPT}

============================================================
PROCESS STRUCTURED TREE
============================================================

{pst}

============================================================
DETECTED VIOLATION
============================================================

{json.dumps(target_violation, indent=2)}

============================================================
RESOLUTION CONTEXT REQUIREMENTS
============================================================

These requirements are currently satisfied and should remain
satisfied whenever possible.

Avoid introducing new violations of these requirements.

{json.dumps(resolution_context, indent=2)}

IMPORTANT:
- Return ONLY valid JSON
- Do not use markdown
- Do not use code fences
- Ensure output is parseable with json.loads()
"""

    # ========================================================
    # API REQUEST
    # ========================================================

    print(
        "\nSending request to model...\n"
    )

    try:

        response = requests.post(
            url=(
                "https://openrouter.ai/"
                "api/v1/chat/completions"
            ),
            headers={
                "Authorization": (
                    f"Bearer "
                    f"{OPENROUTER_API_KEY}"
                ),
                "Content-Type":
                    "application/json",
            },
            data=json.dumps({
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": final_prompt
                    }
                ],
                "temperature": 0,
                "reasoning": {
                    "enabled": True
                }
            }),
            timeout=300
        )

    except Exception as e:

        problems_detected.append(
            f"Request exception: {str(e)}"
        )

        raise e

    # ========================================================
    # CHECK RESPONSE
    # ========================================================

    if response.status_code != 200:

        problems_detected.append(
            f"HTTP {response.status_code}"
        )

        print("ERROR:")

        print(response.status_code)

        print(response.text)

        raise Exception(
            "API request failed"
        )

    response_json = response.json()

    # ========================================================
    # EXTRACT MODEL OUTPUT
    # ========================================================

    print(
        "\n========== FULL RESPONSE ==========\n"
    )

    print(
        json.dumps(
            response_json,
            indent=2
        )
    )

    if "choices" not in response_json:

        problems_detected.append(
            "API response missing 'choices'"
        )

        raise ValueError(
            "API response does not contain "
            "'choices'."
        )

    try:

        assistant_message = (
            response_json["choices"][0]["message"]
        )

        generated_text = (
            assistant_message["content"]
        )

    except Exception as e:

        problems_detected.append(
            "Failed to extract assistant message"
        )

        raise e

    # ========================================================
    # VALIDATE JSON
    # ========================================================

    parsed_json = None

    try:

        parsed_json = json.loads(
            generated_text
        )

    except Exception as e:

        problems_detected.append(
            "Model output is not valid JSON"
        )

        print(
            "\nERROR: Invalid JSON generated"
        )

        print(e)

    # ========================================================
    # SAVE RAW RESPONSE
    # ========================================================

    raw_output_path = (
        RAW_DIR
        / f"{OUTPUT_PREFIX}_raw.txt"
    )

    with open(
        raw_output_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(generated_text)

    print(
        f"\nRaw response saved to:\n"
        f"{raw_output_path}"
    )

    # ========================================================
    # SAVE PARSED JSON
    # ========================================================

    if parsed_json is not None:

        json_output_path = (
            JSON_DIR
            / f"{OUTPUT_PREFIX}.json"
        )

        with open(
            json_output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                parsed_json,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\nJSON saved to:\n"
            f"{json_output_path}"
        )

    # ========================================================
    # SAVE FULL API RESPONSE
    # ========================================================

    full_response_path = (
        FULL_RESPONSE_DIR
        / (
            f"{OUTPUT_PREFIX}"
            f"_full_response.json"
        )
    )

    with open(
        full_response_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            response_json,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nFull API response saved to:\n"
        f"{full_response_path}"
    )

    # ========================================================
    # EXECUTION TIME
    # ========================================================

    end_time = time.time()

    execution_time_ms = round(
        (end_time - start_time) * 1000,
        2
    )

    execution_time_sec = round(
        (end_time - start_time),
        2
    )

    # ========================================================
    # TOKEN USAGE
    # ========================================================

    usage = response_json.get(
        "usage",
        {}
    )

    # ========================================================
    # SAVE METADATA
    # ========================================================

    metadata = {

        "model":
            MODEL,

        "scenario_name":
            scenario_name,

        "requirement_id":
            requirement_id,

        "output_prefix":
            OUTPUT_PREFIX,

        "execution_time_milliseconds":
            execution_time_ms,

        "execution_time_seconds":
            execution_time_sec,

        "resolution_context_requirements":
            len(resolution_context),

        "problems_detected":
            problems_detected,

        "usage": {

            "prompt_tokens":
                usage.get(
                    "prompt_tokens"
                ),

            "completion_tokens":
                usage.get(
                    "completion_tokens"
                ),

            "total_tokens":
                usage.get(
                    "total_tokens"
                ),

            "reasoning_tokens":
                usage.get(
                    "reasoning_tokens"
                )
        }
    }

    metadata_path = (
        METADATA_DIR
        / (
            f"{OUTPUT_PREFIX}"
            f"_metadata.json"
        )
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"\nMetadata saved to:\n"
        f"{metadata_path}"
    )

    # ========================================================
    # SAVE PROBLEM LOG
    # ========================================================

    problem_log_path = (
        PROBLEMS_DIR
        / (
            f"{OUTPUT_PREFIX}"
            f"_problems.log"
        )
    )

    with open(
        problem_log_path,
        "w",
        encoding="utf-8"
    ) as f:

        if len(problems_detected) == 0:

            f.write(
                "No problems detected.\n"
            )

        else:

            for problem in problems_detected:

                f.write(problem + "\n")

    print(
        f"\nProblem log saved to:\n"
        f"{problem_log_path}"
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    print(
        "\n========== EXECUTION SUMMARY ==========\n"
    )

    print(
        "Execution time (milliseconds):",
        execution_time_ms
    )

    print(
        "Execution time (seconds):",
        execution_time_sec
    )

    print(
        "Prompt tokens:",
        usage.get("prompt_tokens")
    )

    print(
        "Completion tokens:",
        usage.get("completion_tokens")
    )

    print(
        "Total tokens:",
        usage.get("total_tokens")
    )

    print(
        "Reasoning tokens:",
        usage.get("reasoning_tokens")
    )

    print(
        "Resolution context requirements:",
        len(resolution_context)
    )

    print("\nProblems detected:")

    if len(problems_detected) == 0:

        print("None")

    else:

        for problem in problems_detected:

            print("-", problem)

# ============================================================
# GLOBAL ITERATION
# ============================================================

all_failures = []

scenario_execution_times = {}

global_start_time = time.time()

for scenario_name, requirement_ids in SCENARIOS.items():

    print("\n================================================")
    print(f"STARTING SCENARIO: {scenario_name}")
    print("================================================")

    scenario_start_time = time.time()

    for requirement_id in requirement_ids:

        try:

            run_generation(
                scenario_name,
                requirement_id
            )

        except Exception as e:

            failure = {

                "scenario":
                    scenario_name,

                "requirement":
                    requirement_id,

                "error":
                    str(e)
            }

            all_failures.append(
                failure
            )

            print("\nERROR DETECTED")

            print(
                json.dumps(
                    failure,
                    indent=2
                )
            )

    # ========================================================
    # SCENARIO TIME
    # ========================================================

    scenario_end_time = time.time()

    scenario_execution_time_sec = round(
        scenario_end_time
        - scenario_start_time,
        2
    )

    scenario_execution_times[
        scenario_name
    ] = scenario_execution_time_sec

    print("\n------------------------------------------------")
    print(
        f"Scenario finished: "
        f"{scenario_name}"
    )

    print(
        f"Execution time: "
        f"{scenario_execution_time_sec} seconds"
    )

    print("------------------------------------------------")

# ============================================================
# FINAL SUMMARY
# ============================================================

global_end_time = time.time()

global_execution_time_sec = round(
    global_end_time
    - global_start_time,
    2
)

print("\n================================================")
print("BATCH EXECUTION FINISHED")
print("================================================")

print(
    f"\nTotal execution time: "
    f"{global_execution_time_sec} seconds"
)

print("\n================================================")
print("TIME PER SCENARIO")
print("================================================")

for scenario_name, execution_time in (
    scenario_execution_times.items()
):

    print(
        f"{scenario_name}: "
        f"{execution_time} seconds"
    )

print(
    f"\nTotal failures: "
    f"{len(all_failures)}"
)

if len(all_failures) > 0:

    print("\nFAILURES:\n")

    for failure in all_failures:

        print(
            json.dumps(
                failure,
                indent=2
            )
        )


In [3]:
# Considering external resolution context
# ============================================================
# STEP 2:
# APPLY AND VALIDATE RESOLUTION STRATEGIES
# ============================================================
#
# This script applies the generated resolution strategies to
# the original process models and validates the resulting
# updated Process Structure Trees (PSTs).
#
# For each scenario and violated requirement, the script:
# - loads the original process model
# - loads the generated resolution strategies
# - applies each change operation sequentially
# - validates the updated process structure
# - stores the updated PSTs and execution logs
#
# The validation pipeline includes:
# - change operation validation
# - behavioral validation
# - PST validation
# - structural validation
#
# The script also generates:
# - a global CSV summary
# - validation statistics by scenario
#
# Output:
# - updated PST models (.xml)
# - validation and execution logs
# - validation summary CSV files
#
# ============================================================


# ============================================================
# STEP 2:
# APPLY AND VALIDATE RESOLUTION STRATEGIES
# ============================================================

import sys
import json
import time
import csv
from pathlib import Path
from collections import defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_PATH = CURRENT_DIR

while BASE_PATH.name != (
    "PTResolver"
):
    BASE_PATH = BASE_PATH.parent

print("Project root:")
print(BASE_PATH)

# ============================================================
# EXECUTION MODE
# ============================================================

RUN_ALL = True

SCENARIO_NAME = ""

REQUIREMENT_ID = ""

# ============================================================
# SCENARIO / REQUIREMENT CONFIGURATION
# ============================================================

SCENARIO_REQUIREMENTS = {

    "01_awad_delivery_of_goods":
      ["R1", "R2", "R3", "R4"],

    "02_de_masellis_loan_approval":
       ["R1", "R2", "R3"],

    "03_elgammal_loan_approval":
       ["R2", "R3"],

    "04_ghose_package_screening":
        ["R1"],

   "05_loebbecke_blood_collection":
        ["R6"],

    "06_BPMQ":
        ["R2", "R4", "R5", "R6"],

     "07_CRL":
        ["R1", "R3", "R4", "R6"],

     "08_DCR":
        ["R2"],

    "09_HaarmannetAL2021":
        ["R1"],

    "11_PCL":
        ["R1", "R3", "R4"],

    "12_PPSL":
        ["R2", "R3", "R4"],

    "13_Status":
        ["R1", "R2"],

    "14_SunetAl24":
        ["R2", "R3", "R4"],

    "15_WinteretAlER":
        ["R2", "R3"],

    "16_Zasadaetal23":
        ["R2", "R3", "R5"]
}


# ============================================================
# BUILD EXECUTION LIST
# ============================================================

EXECUTION_LIST = []

if RUN_ALL:

    for scenario, requirements in (
        SCENARIO_REQUIREMENTS.items()
    ):

        for req_id in requirements:

            EXECUTION_LIST.append(
                (scenario, req_id)
            )

else:

    EXECUTION_LIST.append(
        (
            SCENARIO_NAME,
            REQUIREMENT_ID
        )
    )

# ============================================================
# PYTHON PATHS
# ============================================================

SRC_DIR = (
    BASE_PATH
    / "src"
)

STEP2_DIR = (
    SRC_DIR
    / "step_2_apply_validate_resolution_strategies"
)

sys.path.append(str(SRC_DIR))
sys.path.append(str(STEP2_DIR))

# ============================================================
# IMPORTS
# ============================================================

from utils.process_io import load_process, save_process

from change_management.change_applier import (
    ChangeApplier
)

from validators.change_operation_validator import (
    ChangeOperationValidator
)

from validators.behavioral_validator import (
    BehavioralValidator
)

from validators.pst_validator import (
    PSTValidator
)

from validators.structural_validator import (
    StructuralValidator
)

from change_operations.operations import *

# ============================================================
# CHANGE OPERATION MAPPING
# ============================================================

operation_mapping = {
    "insert_after": insert_after,
    "insert_before": insert_before,
    "delete": delete,
    "rename": rename,
    "move_after": move_after,
    "move_before": move_before,
    "swap": swap,
    "merge": merge_by_label,
    "split": split,
    "copy_after": copy_after,
    "copy_before": copy_before,
    "modify_condition": modify_condition,
    "modify_resource": modify_resource,
    "modify_write": modify_write,
    "add_write": add_write,
    "remove_write": remove_write,
    "modify_read": modify_read,
    "parallelize": parallelize,
    "sequentialize_parallel": sequentialize_parallel,
    "add_xor_branch": add_xor_branch,
    "remove_branch": remove_branch,
    "remove_branch_by_condition":
        remove_branch_by_condition,
    "embed_activity_in_xor":
        embed_activity_in_xor,
    "embed_pre_loop": embed_pre_loop,
    "embed_post_loop": embed_post_loop,
    "remove_loop": remove_loop,
    "modify_loop_condition":
        modify_loop_condition,
    "modify_timeout": modify_timeout
}

# ============================================================
# VALIDATORS + APPLIER
# ============================================================

applier = ChangeApplier()

pattern_validator = ChangeOperationValidator()

behavioral_validator = BehavioralValidator()

pst_validator = PSTValidator()

structural_validator = StructuralValidator()

# ============================================================
# GLOBAL VALIDATION SUMMARY
# ============================================================

validation_summary = []

scenario_statistics = defaultdict(
    lambda: {
        "total_strategies": 0,
        "applied": 0,
        "warnings": 0,
        "errors": 0
    }
)

# ============================================================
# EXECUTION
# ============================================================

for (
    SCENARIO_NAME,
    REQUIREMENT_ID
) in EXECUTION_LIST:

    print("\n================================================")
    print("SCENARIO:", SCENARIO_NAME)
    print("REQUIREMENT:", REQUIREMENT_ID)
    print("================================================")

    ORIGINAL_MODEL_PATH = (
        BASE_PATH
        / "data/input/process_models/cpee_trees"
        / f"{SCENARIO_NAME}.xml"
    )

    RESOLUTION_STRATEGIES_FILE = (
        BASE_PATH
        / "data/ablation_study_step_1/final_prompt_4th_iteration/generated_resolution_strategies"
        / SCENARIO_NAME
        / "resolution_strategies_clean"
        / f"{SCENARIO_NAME}_RS_{REQUIREMENT_ID}.json"
    )

    if not RESOLUTION_STRATEGIES_FILE.exists():

        print(
            "Resolution strategy file missing:"
        )

        print(RESOLUTION_STRATEGIES_FILE)

        continue

    SCENARIO_OUTPUT_DIR = (
        BASE_PATH
        / "data/ablation_study_step_1/final_prompt_4th_iteration/updated_pst"
        / SCENARIO_NAME
    )

    PST_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "pst"
    )

    LOG_OUTPUT_DIR = (
        SCENARIO_OUTPUT_DIR
        / "logs"
    )

    PST_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    LOG_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        RESOLUTION_STRATEGIES_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        resolution_data = json.load(f)

    resolution_strategies = (
        resolution_data[
            "resolution_strategies"
        ]
    )

    for strategy in resolution_strategies:

        strategy_id = strategy[
            "resolution_strategy_id"
        ]

        print("\n------------------------------------------------")
        print("Applying strategy:", strategy_id)
        print("------------------------------------------------")

        tree, root = load_process(
            str(ORIGINAL_MODEL_PATH)
        )

        current_root = root

        strategy_logs = []

        applied_operations = []

        strategy_start_time = time.time()

        strategy_applied = "YES"

        validation_status = "SUCCESS"

        warnings_detected = "NO"

        change_operation_validation = (
            "SUCCESS"
        )

        behavioral_validation = (
            "NOT_EXECUTED"
        )

        pst_validation = (
            "NOT_EXECUTED"
        )

        structural_validation = (
            "NOT_EXECUTED"
        )

        failure_message = ""

        for operation_data in strategy[
            "change_operations"
        ]:

            operation_name = operation_data[
                "operation"
            ]

            parameters = operation_data[
                "parameters"
            ]

            print(
                "\nOperation:",
                operation_name
            )

            print(
                "Parameters:",
                parameters
            )

            if (
                operation_name
                not in operation_mapping
            ):

                raise ValueError(
                    f"Unsupported operation: "
                    f"{operation_name}"
                )

            operation_function = (
                operation_mapping[
                    operation_name
                ]
            )

            pattern_validator.validate(
                operation_function
            )

            strategy_logs.append(
                f"ChangeOperationValidator: "
                f"SUCCESS ({operation_name})"
            )

            args = []

            # =================================================
            # ARGUMENT MAPPING
            # =================================================

            if operation_name in [
                "insert_after",
                "insert_before"
            ]:

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == "delete":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "rename":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name in [
                "move_after",
                "move_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == "swap":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "merge":

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "split":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "copy_after",
                "copy_before"
            ]:

                args = [
                    parameters[
                        "source_activity_label"
                    ],
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name in [
                "parallelize",
                "sequentialize_parallel"
            ]:

                args = [
                    parameters[
                        "first_activity_label"
                    ],
                    parameters[
                        "second_activity_label"
                    ]
                ]

            elif operation_name == "remove_branch":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "remove_branch_by_condition"
            ):

                args = [
                    parameters[
                        "target_condition"
                    ]
                ]

            elif operation_name == (
                "modify_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_resource"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_resource"
                    ]
                ]

            elif operation_name == (
                "modify_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_statement"
                    ]
                ]

            elif operation_name == "add_write":

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "added_statement"
                    ]
                ]

            elif operation_name == (
                "remove_write"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "removed_variable_name"
                    ]
                ]

            elif operation_name == (
                "modify_read"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "old_variable_name"
                    ],
                    parameters[
                        "new_variable_name"
                    ]
                ]

            elif operation_name == "remove_loop":

                args = [
                    parameters[
                        "target_activity_label"
                    ]
                ]

            elif operation_name == (
                "modify_loop_condition"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_condition"
                    ]
                ]

            elif operation_name == (
                "modify_timeout"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "new_timeout"
                    ]
                ]

            elif operation_name == (
                "add_xor_branch"
            ):

                args = [
                    parameters[
                        "existing_branch_condition"
                    ],
                    parameters[
                        "new_branch_condition"
                    ],
                    parameters[
                        "new_activity_label"
                    ]
                ]

            elif operation_name == (
                "embed_activity_in_xor"
            ):

                args = [
                    parameters[
                        "target_activity_label"
                    ],
                    parameters[
                        "condition"
                    ],
                    parameters[
                        "mode"
                    ]
                ]

            elif operation_name in [
                "embed_pre_loop",
                "embed_post_loop"
            ]:

                args = [
                    parameters[
                        "start_activity_label"
                    ],
                    parameters[
                        "end_activity_label"
                    ],
                    parameters[
                        "condition"
                    ]
                ]

            else:

                raise ValueError(
                    f"Argument mapping missing "
                    f"for operation: "
                    f"{operation_name}"
                )

            # =================================================
            # APPLY CHANGE
            # =================================================

            try:

                current_root, log = (
                    applier.apply(
                        current_root,
                        operation_function,
                        *args
                    )
                )

                applied_operations.append(
                    f"{operation_name}"
                )

                strategy_logs.append(log)

            except Exception as e:

                print("FAILED TO APPLY CHANGE")

                print(str(e))

                strategy_applied = "NO"

                validation_status = "ERROR"

                failure_message = str(e)

                strategy_logs.append(
                    f"APPLICATION FAILED: {str(e)}"
                )

                break

            # =================================================
            # VALIDATION PHASE
            # =================================================

            try:

                behavioral_validator.validate(
                    current_root
                )

                behavioral_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "BehavioralValidator: SUCCESS"
                )

            except Exception as e:

                behavioral_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"BehavioralValidator WARNING: "
                    f"{str(e)}"
                )

            try:

                pst_validator.validate(
                    current_root
                )

                pst_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "PSTValidator: SUCCESS"
                )

            except Exception as e:

                pst_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"PSTValidator WARNING: "
                    f"{str(e)}"
                )

            try:

                structural_warnings = (
                    structural_validator.validate(
                        current_root
                    )
                )

                structural_validation = (
                    "SUCCESS"
                )

                strategy_logs.append(
                    "StructuralValidator: SUCCESS"
                )

                if structural_warnings:

                    warnings_detected = (
                        "YES"
                    )

                    validation_status = (
                        "WARNING"
                    )

                    for warning in (
                        structural_warnings
                    ):

                        strategy_logs.append(
                            f"Structural WARNING: "
                            f"{warning}"
                        )

            except Exception as e:

                structural_validation = (
                    "WARNING"
                )

                validation_status = (
                    "WARNING"
                )

                warnings_detected = "YES"

                strategy_logs.append(
                    f"StructuralValidator WARNING: "
                    f"{str(e)}"
                )

            print("SUCCESS")

        # =====================================================
        # EXECUTION TIME
        # =====================================================

        strategy_end_time = time.time()

        execution_time_ms = round(
            (
                strategy_end_time
                - strategy_start_time
            ) * 1000,
            2
        )

        strategy_logs.append(
            f"Execution time "
            f"(milliseconds): "
            f"{execution_time_ms}"
        )

        # =====================================================
        # SAVE VALIDATION SUMMARY
        # =====================================================

        validation_summary.append({

            "scenario":
                SCENARIO_NAME,

            "requirement_id":
                REQUIREMENT_ID,

            "strategy_id":
                strategy_id,

            "strategy_applied":
                strategy_applied,

            "validation_status":
                validation_status,

            "warnings_detected":
                warnings_detected,

            "change_operation_validator":
                change_operation_validation,

            "behavioral_validator":
                behavioral_validation,

            "pst_validator":
                pst_validation,

            "structural_validator":
                structural_validation,

            "applied_operations":
                " | ".join(
                    applied_operations
                ),

            "execution_time_ms":
                execution_time_ms,

            "failure_message":
                failure_message
        })

        # =====================================================
        # SCENARIO STATISTICS
        # =====================================================

        scenario_statistics[
            SCENARIO_NAME
        ]["total_strategies"] += 1

        if strategy_applied == "YES":

            scenario_statistics[
                SCENARIO_NAME
            ]["applied"] += 1

        if validation_status == "WARNING":

            scenario_statistics[
                SCENARIO_NAME
            ]["warnings"] += 1

        if validation_status == "ERROR":

            scenario_statistics[
                SCENARIO_NAME
            ]["errors"] += 1

        # =====================================================
        # SAVE UPDATED MODEL
        # =====================================================

        tree._setroot(current_root)

        output_model_path = (
            PST_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}.xml"
            )
        )

        save_process(
            tree,
            str(output_model_path)
        )

        print("\nSaved updated model:")

        print(output_model_path)

        # =====================================================
        # SAVE LOG
        # =====================================================

        log_path = (
            LOG_OUTPUT_DIR
            / (
                f"{SCENARIO_NAME}_"
                f"{REQUIREMENT_ID}_"
                f"{strategy_id}_log.txt"
            )
        )

        with open(
            log_path,
            "w",
            encoding="utf-8"
        ) as f:

            for log_entry in strategy_logs:

                f.write(str(log_entry))
                f.write("\n")

        print("Saved log:")

        print(log_path)

# ============================================================
# SAVE GLOBAL CSV SUMMARY
# ============================================================

summary_csv_path = (
    BASE_PATH
    / "data/output/updated_pst"
    / "validation_summary.csv"
)

with open(
    summary_csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[

            "scenario",
            "requirement_id",
            "strategy_id",

            "strategy_applied",
            "validation_status",
            "warnings_detected",

            "change_operation_validator",
            "behavioral_validator",
            "pst_validator",
            "structural_validator",

            "applied_operations",

            "execution_time_ms",

            "failure_message"
        ]
    )

    writer.writeheader()

    for row in validation_summary:

        writer.writerow(row)

print("\nSaved global validation summary:")

print(summary_csv_path)

# ============================================================
# SAVE SCENARIO STATISTICS CSV
# ============================================================

scenario_stats_csv = (
    BASE_PATH
    / "data/output/updated_pst"
    / "scenario_statistics.csv"
)

with open(
    scenario_stats_csv,
    "w",
    newline="",
    encoding="utf-8"
) as csvfile:

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[

            "scenario",

            "total_strategies",

            "applied",

            "warnings",

            "errors",

            "application_rate"
        ]
    )

    writer.writeheader()

    for (
        scenario,
        stats
    ) in scenario_statistics.items():

        application_rate = round(
            (
                stats["applied"]
                / stats["total_strategies"]
            ) * 100,
            2
        )

        writer.writerow({

            "scenario":
                scenario,

            "total_strategies":
                stats["total_strategies"],

            "applied":
                stats["applied"],

            "warnings":
                stats["warnings"],

            "errors":
                stats["errors"],

            "application_rate":
                application_rate
        })

print("\nSaved scenario statistics:")

print(scenario_stats_csv)

print("\n================================================")
print("ALL STRATEGIES COMPLETED")
print("================================================")

Project root:
/home/marisolbarrientosmoreno/Desktop/BPM_2026_mitigation_actions/PTResolver

SCENARIO: 01_awad_delivery_of_goods
REQUIREMENT: R1

------------------------------------------------
Applying strategy: RS1_copy_notification_to_bank_transfer_branch
------------------------------------------------

Operation: copy_after
Parameters: {'source_activity_label': 'notify customer', 'target_activity_label': 'pay by bank transfer'}
SUCCESS

Saved updated model:
/home/marisolbarrientosmoreno/Desktop/BPM_2026_mitigation_actions/PTResolver/data/ablation_study_step_1/final_prompt_4th_iteration/updated_pst/01_awad_delivery_of_goods/pst/01_awad_delivery_of_goods_R1_RS1_copy_notification_to_bank_transfer_branch.xml
Saved log:
/home/marisolbarrientosmoreno/Desktop/BPM_2026_mitigation_actions/PTResolver/data/ablation_study_step_1/final_prompt_4th_iteration/updated_pst/01_awad_delivery_of_goods/logs/01_awad_delivery_of_goods_R1_RS1_copy_notification_to_bank_transfer_branch_log.txt

------------

In [13]:
# Considering external resolution context
# ============================================================
# STEP 3: VERIFY COMPLIANCE OF UPDATED PSTS
# ============================================================
#
# This script evaluates the updated Process Structure Trees
# (PSTs) generated after applying resolution strategies.
#
# The verification now includes:
# - originally violated requirements
# - resolution context requirements
#
# This allows checking:
# - whether the targeted violations were fixed
# - whether previously compliant requirements
#   remain compliant after adaptation
#
# Output:
# - compliance violation reports for updated PSTs (.json)
#
# ============================================================

import json
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

# -----------------------------------
# Configuration
# -----------------------------------

RUN_ALL_SCENARIOS = True

SELECTED_SCENARIO = (
    "01_awad_delivery_of_goods"
)

VERIFY_ALL = True

selected_requirement_ids = [
    "R1"
]

# -----------------------------------
# Dynamically locate project root
# -----------------------------------

BASE_DIR = Path.cwd().parents[1]

# -----------------------------------
# Paths
# -----------------------------------

PTV_SCRIPT = (
    BASE_DIR
    / "src"
    / "ProcessTreeVerify"
    / "python_code"
    / "test_script.py"
)

UPDATED_PST_DIR = (
    BASE_DIR
    / "data"
    / "ablation_study_step_1"
    / "final_prompt_4th_iteration"
    / "updated_pst"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "ablation_study_step_1"
    / "final_prompt_4th_iteration"
    / "compliance_violations_after_changes"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -----------------------------------
# Requirement folders
# -----------------------------------

VIOLATION_REQUIREMENTS_DIR = (
    BASE_DIR
    / "data"
    / "input"
    / "compliance_requirements"
    / "ast"
)

RESOLUTION_CONTEXT_DIR = (
    BASE_DIR
    / "data"
    / "input"
    / "resolution_context"
)

# -----------------------------------
# Namespaces
# -----------------------------------

NS_PROPERTIES = (
    "http://cpee.org/ns/properties/2.0"
)

NS_SUBSCRIPTIONS = (
    "http://riddl.org/ns/common-patterns/"
    "notifications-producer/2.0"
)

namespace = {
    "ns1": NS_PROPERTIES
}

# -----------------------------------
# Ensure <requirements> node
# -----------------------------------

def ensure_requirements_node(root):

    attributes_node = root.find(
        ".//ns1:attributes",
        namespace
    )

    if attributes_node is None:

        raise ValueError(
            "No <attributes> node found."
        )

    req_node = root.find(
        ".//ns1:requirements",
        namespace
    )

    if req_node is None:

        print(
            "\n[INFO] Creating "
            "<requirements> node."
        )

        req_node = ET.SubElement(
            attributes_node,
            f"{{{NS_PROPERTIES}}}requirements"
        )

    else:

        print(
            "\n[INFO] Existing "
            "<requirements> node found."
        )

    return req_node

# -----------------------------------
# Ensure <subscriptions> block
# -----------------------------------

def ensure_subscriptions(root):

    subscriptions = root.find(
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    if subscriptions is not None:

        print(
            "\n[INFO] Subscriptions block already exists."
        )

        return

    print(
        "\n[INFO] Creating subscriptions block."
    )

    subscriptions = ET.SubElement(
        root,
        f"{{{NS_SUBSCRIPTIONS}}}subscriptions"
    )

    subscription = ET.SubElement(
        subscriptions,
        f"{{{NS_SUBSCRIPTIONS}}}subscription",
        {
            "id": "_compliance",
            "url": (
                "https://power.bpm.cit.tum.de/"
                "compliance/Subscriber"
            )
        }
    )

    topic = ET.SubElement(
        subscription,
        f"{{{NS_SUBSCRIPTIONS}}}topic",
        {
            "id": "description"
        }
    )

    event = ET.SubElement(
        topic,
        f"{{{NS_SUBSCRIPTIONS}}}event"
    )

    event.text = "change"

# -----------------------------------
# Load resolution context
# -----------------------------------

def load_resolution_context(
    scenario_name
):

    resolution_context_file = (
        RESOLUTION_CONTEXT_DIR
        / (
            f"{scenario_name}"
            f"_req_resolution_context.json"
        )
    )

    if not resolution_context_file.exists():

        print(
            "\n[INFO] No resolution context "
            "requirements found."
        )

        return {}

    with open(
        resolution_context_file,
        "r",
        encoding="utf-8"
    ) as f:

        resolution_context = json.load(f)

    print(
        f"\n[INFO] Loaded "
        f"{len(resolution_context)} "
        f"resolution context requirements."
    )

    return resolution_context

# -----------------------------------
# Run one PST
# -----------------------------------

def run_pst(
    scenario_name,
    pst_file
):

    print("\n===================================")
    print(f"SCENARIO: {scenario_name}")
    print(f"PST: {pst_file.name}")
    print("===================================\n")

    # -----------------------------------
    # Load violation requirements
    # -----------------------------------

    requirements_file = (
        VIOLATION_REQUIREMENTS_DIR
        / f"{scenario_name}.json"
    )

    with open(
        requirements_file,
        "r",
        encoding="utf-8"
    ) as f:

        all_requirements = json.load(f)

    print(
        f"\n[INFO] Loaded "
        f"{len(all_requirements)} "
        f"violation requirements."
    )

    # -----------------------------------
    # Select requirements
    # -----------------------------------

    if VERIFY_ALL:

        requirements = all_requirements

        print(
            "\nMode: VERIFY ALL VIOLATION REQUIREMENTS"
        )

    else:

        print(
            "\nMode: VERIFY SELECTED REQUIREMENTS"
        )

        missing_requirements = [
            rid
            for rid in selected_requirement_ids
            if rid not in all_requirements
        ]

        if missing_requirements:

            raise ValueError(
                f"Requirements not found: "
                f"{missing_requirements}"
            )

        requirements = {
            rid: all_requirements[rid]
            for rid in selected_requirement_ids
        }

    # -----------------------------------
    # Load resolution context
    # -----------------------------------

    resolution_context = (
        load_resolution_context(
            scenario_name
        )
    )

    # -----------------------------------
    # Merge requirements
    # -----------------------------------

    merged_requirements = {
        **requirements,
        **resolution_context
    }

    requirements_text = json.dumps(
        merged_requirements,
        ensure_ascii=False
    )

    print("\nInjected requirements:")
    print(requirements_text)

    print(
        f"\n[INFO] Total injected requirements:"
        f" {len(merged_requirements)}"
    )

    # -----------------------------------
    # Parse PST
    # -----------------------------------

    tree = ET.parse(pst_file)

    root = tree.getroot()

    req_node = ensure_requirements_node(root)

    ensure_subscriptions(root)

    req_node.text = requirements_text

    print(
        "\n[INFO] Requirements successfully injected."
    )

    # -----------------------------------
    # Create temp XML
    # -----------------------------------

    with tempfile.NamedTemporaryFile(
        suffix=".xml",
        delete=False
    ) as tmp:

        temp_xml_path = Path(tmp.name)

    tree.write(
        temp_xml_path,
        encoding="utf-8",
        xml_declaration=True
    )

    print(
        f"\nTemporary XML created:\n"
        f"{temp_xml_path}"
    )

    # -----------------------------------
    # Run verifier
    # -----------------------------------

    result = subprocess.run(
        [
            "python3",
            str(PTV_SCRIPT),
            str(temp_xml_path)
        ],
        capture_output=True,
        text=True
    )

    print("\n=== STDOUT ===")
    print(result.stdout)

    print("\n=== STDERR ===")
    print(result.stderr)

    # -----------------------------------
    # Extract violation report
    # -----------------------------------

    marker = "===== VIOLATION REPORT ====="

    if marker in result.stdout:

        violation_json = (
            result.stdout
            .split(marker)[-1]
            .strip()
        )

        violation_report = json.loads(
            violation_json
        )

        # -----------------------------------
        # Add metadata
        # -----------------------------------

        violation_report_with_metadata = {

            "scenario_name":
                scenario_name,

            "pst_file":
                pst_file.name,

            "verified_violation_requirements":
                list(requirements.keys()),

            "verified_resolution_context_requirements":
                list(
                    resolution_context.keys()
                ),

            "all_verified_requirements":
                list(
                    merged_requirements.keys()
                ),

            "violation_report":
                violation_report
        }

        # -----------------------------------
        # Scenario output directory
        # -----------------------------------

        scenario_output_dir = (
            OUTPUT_DIR
            / scenario_name
        )

        scenario_output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        # -----------------------------------
        # Output filename
        # -----------------------------------

        output_file = (
            scenario_output_dir
            / (
                f"{pst_file.stem}"
                f"_violations.json"
            )
        )

        with open(
            output_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                violation_report_with_metadata,
                f,
                indent=4,
                ensure_ascii=False
            )

        print(
            f"\nViolation report saved to:\n"
            f"{output_file}"
        )

    else:

        print(
            "\nNo violation report found."
        )

    # -----------------------------------
    # Delete temp XML
    # -----------------------------------

    temp_xml_path.unlink(
        missing_ok=True
    )

    print(
        "\nTemporary XML deleted."
    )

# -----------------------------------
# Main execution
# -----------------------------------

scenario_dirs = sorted(
    UPDATED_PST_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    if (
        not RUN_ALL_SCENARIOS
        and scenario_name != SELECTED_SCENARIO
    ):
        continue

    pst_dir = (
        scenario_dir
        / "pst"
    )

    if not pst_dir.exists():

        print(
            f"\nNo pst folder found for:"
            f" {scenario_name}"
        )

        continue

    pst_files = sorted(
        pst_dir.glob("*.xml")
    )

    print("\n===================================")
    print(f"FOUND {len(pst_files)} PST FILES")
    print(f"FOR SCENARIO {scenario_name}")
    print("===================================\n")

    for pst_file in pst_files:

        run_pst(
            scenario_name,
            pst_file
        )



FOUND 15 PST FILES
FOR SCENARIO 01_awad_delivery_of_goods


SCENARIO: 01_awad_delivery_of_goods
PST: 01_awad_delivery_of_goods_R1_RS1_copy_notification_to_bank_transfer_branch.xml


[INFO] Loaded 4 violation requirements.

Mode: VERIFY ALL VIOLATION REQUIREMENTS

[INFO] No resolution context requirements found.

Injected requirements:
{"R1": "leads_to('go to checkout', 'notify customer')", "R2": "leads_to('pay by credit card', 'pack goods') and leads_to('pay by bank transfer', 'pack goods')", "R3": "leads_to('pack goods', 'notify customer')", "R4": "exists('archive order') and directly_follows('archive order', 'terminate')"}

[INFO] Total injected requirements: 4

[INFO] Creating <requirements> node.

[INFO] Creating subscriptions block.

[INFO] Requirements successfully injected.

Temporary XML created:
/tmp/tmpgd48hfjx.xml

=== STDOUT ===
exists_by_label: no match found for 'archive order'
Available labels:
 - 'go to checkout'
 - 'provide bank data'
 - 'pay by bank transfer'
 - 'not

In [16]:
# ============================================================
# ANALYSIS OF RESOLVED AND UNRESOLVED VIOLATIONS
# ============================================================
#
# This script analyzes whether the generated resolution
# strategies successfully resolved the targeted compliance
# violations in the updated Process Structure Trees (PSTs).
#
# The script identifies:
# - resolved violations
# - unresolved violations
# - requirements never resolved
#
# Output:
# - resolution analysis summaries (.xlsx)
#
# ============================================================

# !pip install openpyxl
import json
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

# Project root
BASE_DIR = Path.cwd().parents[1]

VIOLATIONS_DIR = (
    BASE_DIR
    / "data"
    / "ablation_study_step_1"
    / "final_prompt_4th_iteration"
    / "compliance_violations_after_changes"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "ablation_study_step_1"
    / "final_prompt_4th_iteration"
    / "resolution_strategy_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# RESULTS
# ============================================================

summary_rows = []

# ============================================================
# ITERATE SCENARIOS
# ============================================================

scenario_dirs = sorted(
    VIOLATIONS_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    print("\n===================================")
    print("SCENARIO:", scenario_name)
    print("===================================\n")

    violation_files = sorted(
        scenario_dir.glob("*.json")
    )

    for violation_file in violation_files:

        print("Processing:", violation_file.name)

        # ----------------------------------------------------
        # Extract requirement id from filename
        # ----------------------------------------------------

        # Example:
        # 01_awad_delivery_of_goods_R1_RS1_xxx_violations.json

        stem = violation_file.stem

        parts = stem.split("_")

        requirement_id = None

        for part in parts:

            if part.startswith("R"):

                if (
                    len(part) > 1
                    and part[1:].isdigit()
                ):

                    requirement_id = part
                    break

        if requirement_id is None:

            print(
                "Could not determine "
                "requirement id."
            )

            continue

        # ----------------------------------------------------
        # Load violations
        # ----------------------------------------------------

        with open(
            violation_file,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        # IMPORTANT FIX:
        violations = data.get(
            "violation_report",
            []
        )

        # ----------------------------------------------------
        # Determine if target requirement
        # still violated
        # ----------------------------------------------------

        still_violated = False

        matching_violation = None

        for violation in violations:

            if (
                violation.get("requirement_id")
                == requirement_id
            ):

                still_violated = True

                matching_violation = violation

                break

        # ----------------------------------------------------
        # Determine status
        # ----------------------------------------------------

        if still_violated:

            status = "NOT_FIXED"

            evidence = (
                matching_violation.get(
                    "evidence",
                    []
                )
            )

        else:

            status = "FIXED"

            evidence = []

        # ----------------------------------------------------
        # Add row
        # ----------------------------------------------------

        summary_rows.append({

            "scenario":
                scenario_name,

            "pst_file":
                violation_file.name,

            "requirement_id":
                requirement_id,

            "status":
                status,

            "remaining_evidence":
                " | ".join(evidence)
        })

# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(summary_rows)

# ============================================================
# SORT
# ============================================================

df = df.sort_values(
    by=[
        "scenario",
        "requirement_id",
        "pst_file"
    ]
)

# ============================================================
# REQUIREMENTS NEVER FIXED
# ============================================================

grouped = df.groupby(
    ["scenario", "requirement_id"]
)

never_fixed_rows = []

for (
    scenario,
    requirement_id
), group in grouped:

    statuses = set(
        group["status"].tolist()
    )

    # --------------------------------------------------------
    # requirement fixed at least once
    # --------------------------------------------------------

    if "FIXED" in statuses:

        overall_status = (
            "FIXED_AT_LEAST_ONCE"
        )

    # --------------------------------------------------------
    # never fixed
    # --------------------------------------------------------

    else:

        overall_status = "NEVER_FIXED"

    never_fixed_rows.append({

        "scenario":
            scenario,

        "requirement_id":
            requirement_id,

        "overall_status":
            overall_status,

        "total_strategies":
            len(group),

        "fixed_strategies":
            len(
                group[
                    group["status"]
                    == "FIXED"
                ]
            ),

        "not_fixed_strategies":
            len(
                group[
                    group["status"]
                    == "NOT_FIXED"
                ]
            )
    })

# ============================================================
# CREATE OVERVIEW DATAFRAME
# ============================================================

overview_df = pd.DataFrame(
    never_fixed_rows
)

overview_df = overview_df.sort_values(
    by=[
        "scenario",
        "requirement_id"
    ]
)

# ============================================================
# SAVE EXCEL
# ============================================================

excel_output = (
    OUTPUT_DIR
    / "resolution_strategy_summary.xlsx"
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # detailed results
    # --------------------------------------------------------

    df.to_excel(
        writer,
        sheet_name="all_results",
        index=False
    )

    # --------------------------------------------------------
    # overview
    # --------------------------------------------------------

    overview_df.to_excel(
        writer,
        sheet_name="requirements_overview",
        index=False
    )

    # --------------------------------------------------------
    # only never fixed
    # --------------------------------------------------------

    overview_df[
        overview_df["overall_status"]
        == "NEVER_FIXED"
    ].to_excel(
        writer,
        sheet_name="never_fixed",
        index=False
    )

# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n===================================")
print("SUMMARY")
print("===================================\n")

print(df)

print("\n===================================")
print("REQUIREMENTS OVERVIEW")
print("===================================\n")

print(overview_df)

print("\nExcel saved to:")
print(excel_output)


SCENARIO: 01_awad_delivery_of_goods

Processing: 01_awad_delivery_of_goods_R1_RS1_copy_notification_to_bank_transfer_branch_violations.json
Processing: 01_awad_delivery_of_goods_R1_RS2_insert_notification_after_bank_transfer_payment_violations.json
Processing: 01_awad_delivery_of_goods_R1_RS3_move_notification_before_sending_goods_violations.json
Processing: 01_awad_delivery_of_goods_R1_RS4_remove_bank_transfer_branch_violations.json
Processing: 01_awad_delivery_of_goods_R2_RS1_move_pack_goods_after_parallel_join_violations.json
Processing: 01_awad_delivery_of_goods_R2_RS2_sequentialize_payment_before_shipping_packaging_violations.json
Processing: 01_awad_delivery_of_goods_R2_RS3_add_post_join_pack_goods_before_send_goods_violations.json
Processing: 01_awad_delivery_of_goods_R2_RS4_add_pack_goods_after_each_payment_violations.json
Processing: 01_awad_delivery_of_goods_R3_RS1_move_notify_after_send_goods_violations.json
Processing: 01_awad_delivery_of_goods_R3_RS2_move_notify_after_pac

In [ ]:
# GOOD
# ============================================================
# ANALYSIS OF RESOLVED / UNRESOLVED VIOLATIONS
# + RESOLUTION CONTEXT PRESERVATION
# + VIOLATION EVIDENCE
# ============================================================
#
# This script analyzes:
#
# 1. Whether originally violated requirements
#    are now compliant
#
# 2. Whether previously compliant requirements
#    (resolution context requirements)
#    remain compliant after applying the
#    resolution strategies
#
# 3. Why resolution context requirements
#    became violated
#
# Output:
# - detailed analysis
# - overview statistics
# - preservation analysis
# - Excel summary
#
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# Project root
BASE_DIR = Path(__file__).resolve().parents[2]

VIOLATIONS_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_after_changes"
)

RESOLUTION_CONTEXT_DIR = (
    BASE_DIR
    / "data"
    / "input"
    / "resolution_context"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "output"
    / "resolution_strategy_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)
# ============================================================
# RESULTS
# ============================================================

summary_rows = []

# ============================================================
# ITERATE SCENARIOS
# ============================================================

scenario_dirs = sorted(
    VIOLATIONS_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    print("\n===================================")
    print("SCENARIO:", scenario_name)
    print("===================================\n")

    # --------------------------------------------------------
    # Load resolution context requirements
    # --------------------------------------------------------

    resolution_context_file = (
        RESOLUTION_CONTEXT_DIR
        / (
            f"{scenario_name}"
            f"_req_resolution_context.json"
        )
    )

    resolution_context_requirements = set()

    if resolution_context_file.exists():

        with open(
            resolution_context_file,
            "r",
            encoding="utf-8"
        ) as f:

            resolution_context = json.load(f)

        resolution_context_requirements = set(
            resolution_context.keys()
        )

        print(
            f"Loaded "
            f"{len(resolution_context_requirements)} "
            f"resolution context requirements."
        )

    else:

        print(
            "No resolution context "
            "requirements found."
        )

    # --------------------------------------------------------
    # Violation files
    # --------------------------------------------------------

    violation_files = sorted(
        scenario_dir.glob("*.json")
    )

    for violation_file in violation_files:

        print("Processing:", violation_file.name)

        # ----------------------------------------------------
        # Extract original violated requirement
        # ----------------------------------------------------

        stem = violation_file.stem

        parts = stem.split("_")

        original_requirement_id = None

        for part in parts:

            if (
                part.startswith("R")
                and len(part) > 1
                and part[1:].isdigit()
            ):

                original_requirement_id = part

                break

        if original_requirement_id is None:

            print(
                "Could not determine "
                "requirement id."
            )

            continue

        # ----------------------------------------------------
        # Load violations
        # ----------------------------------------------------

        with open(
            violation_file,
            "r",
            encoding="utf-8"
        ) as f:

            file_content = json.load(f)

        # ----------------------------------------------------
        # Handle new structure
        # ----------------------------------------------------

        if isinstance(file_content, dict):

            violations = file_content.get(
                "violation_report",
                []
            )

        else:

            violations = file_content

        # ----------------------------------------------------
        # Collect violated requirements
        # ----------------------------------------------------

        violated_requirements_after = set()

        for violation in violations:

            if (
                isinstance(violation, dict)
                and "requirement_id" in violation
            ):

                violated_requirements_after.add(
                    violation["requirement_id"]
                )

        # ----------------------------------------------------
        # 1. CHECK IF ORIGINAL VIOLATION FIXED
        # ----------------------------------------------------

        original_fixed = (
            original_requirement_id
            not in violated_requirements_after
        )

        # ----------------------------------------------------
        # 2. CHECK RESOLUTION CONTEXT VIOLATIONS
        # ----------------------------------------------------

        violated_resolution_context = []

        resolution_context_evidence = []

        for violation in violations:

            if not isinstance(violation, dict):
                continue

            violated_req = violation.get(
                "requirement_id"
            )

            if (
                violated_req
                in resolution_context_requirements
            ):

                violated_resolution_context.append(
                    violated_req
                )

                evidence = violation.get(
                    "evidence",
                    []
                )

                if isinstance(evidence, list):

                    evidence_text = " | ".join(
                        str(e)
                        for e in evidence
                    )

                else:

                    evidence_text = str(evidence)

                resolution_context_evidence.append(
                    f"{violated_req}: {evidence_text}"
                )

        violated_resolution_context = sorted(
            set(violated_resolution_context)
        )

        resolution_context_preserved = (
            len(
                violated_resolution_context
            ) == 0
        )

        # ----------------------------------------------------
        # Build status labels
        # ----------------------------------------------------

        if original_fixed:

            original_status = "FIXED"

        else:

            original_status = "NOT_FIXED"

        if resolution_context_preserved:

            context_status = "PRESERVED"

        else:

            context_status = "BROKEN"

        # ----------------------------------------------------
        # Evidence for original violation
        # ----------------------------------------------------

        original_violation_evidence = []

        for violation in violations:

            if not isinstance(violation, dict):
                continue

            if (
                violation.get("requirement_id")
                == original_requirement_id
            ):

                evidence = violation.get(
                    "evidence",
                    []
                )

                if isinstance(evidence, list):

                    original_violation_evidence.extend(
                        evidence
                    )

                else:

                    original_violation_evidence.append(
                        str(evidence)
                    )

        # ----------------------------------------------------
        # Add row
        # ----------------------------------------------------

        summary_rows.append({

            "scenario":
                scenario_name,

            "pst_file":
                violation_file.name,

            "original_requirement":
                original_requirement_id,

            "original_violation_status":
                original_status,

            "original_violation_evidence":
                " || ".join(
                    original_violation_evidence
                ),

            "resolution_context_status":
                context_status,

            "violated_resolution_context_requirements":
                ", ".join(
                    violated_resolution_context
                ),

            "resolution_context_violation_evidence":
                " || ".join(
                    resolution_context_evidence
                ),

            "num_broken_resolution_context_requirements":
                len(
                    violated_resolution_context
                ),

            "all_remaining_violations":
                ", ".join(
                    sorted(
                        violated_requirements_after
                    )
                )
        })

# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(summary_rows)

# ============================================================
# SORT
# ============================================================

df = df.sort_values(
    by=[
        "scenario",
        "original_requirement",
        "pst_file"
    ]
)

# ============================================================
# OVERVIEW
# ============================================================

grouped = df.groupby(
    ["scenario", "original_requirement"]
)

overview_rows = []

for (
    scenario,
    requirement
), group in grouped:

    # --------------------------------------------------------
    # Original violation fixed?
    # --------------------------------------------------------

    fixed_count = len(
        group[
            group[
                "original_violation_status"
            ] == "FIXED"
        ]
    )

    not_fixed_count = len(
        group[
            group[
                "original_violation_status"
            ] == "NOT_FIXED"
        ]
    )

    # --------------------------------------------------------
    # Resolution context preserved?
    # --------------------------------------------------------

    preserved_count = len(
        group[
            group[
                "resolution_context_status"
            ] == "PRESERVED"
        ]
    )

    broken_count = len(
        group[
            group[
                "resolution_context_status"
            ] == "BROKEN"
        ]
    )

    # --------------------------------------------------------
    # Overall statuses
    # --------------------------------------------------------

    if fixed_count > 0:

        overall_fix_status = (
            "FIXED_AT_LEAST_ONCE"
        )

    else:

        overall_fix_status = (
            "NEVER_FIXED"
        )

    if broken_count == 0:

        overall_context_status = (
            "ALWAYS_PRESERVED"
        )

    else:

        overall_context_status = (
            "BROKEN_AT_LEAST_ONCE"
        )

    # --------------------------------------------------------
    # Add row
    # --------------------------------------------------------

    overview_rows.append({

        "scenario":
            scenario,

        "requirement":
            requirement,

        "overall_fix_status":
            overall_fix_status,

        "overall_resolution_context_status":
            overall_context_status,

        "total_strategies":
            len(group),

        "fixed_strategies":
            fixed_count,

        "not_fixed_strategies":
            not_fixed_count,

        "preserved_context_strategies":
            preserved_count,

        "broken_context_strategies":
            broken_count
    })

# ============================================================
# OVERVIEW DATAFRAME
# ============================================================

overview_df = pd.DataFrame(
    overview_rows
)

overview_df = overview_df.sort_values(
    by=[
        "scenario",
        "requirement"
    ]
)

# ============================================================
# SAVE EXCEL
# ============================================================

excel_output = (
    OUTPUT_DIR
    / (
        "resolution_strategy_analysis"
        "_with_context.xlsx"
    )
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    # --------------------------------------------------------
    # Detailed results
    # --------------------------------------------------------

    df.to_excel(
        writer,
        sheet_name="all_results",
        index=False
    )

    # --------------------------------------------------------
    # Overview
    # --------------------------------------------------------

    overview_df.to_excel(
        writer,
        sheet_name="overview",
        index=False
    )

    # --------------------------------------------------------
    # Only unresolved
    # --------------------------------------------------------

    df[
        df[
            "original_violation_status"
        ] == "NOT_FIXED"
    ].to_excel(
        writer,
        sheet_name="not_fixed",
        index=False
    )

    # --------------------------------------------------------
    # Broken resolution context
    # --------------------------------------------------------

    df[
        df[
            "resolution_context_status"
        ] == "BROKEN"
    ].to_excel(
        writer,
        sheet_name="broken_context",
        index=False
    )

# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("SUMMARY")
print("===================================\n")

print(df)

print("\n===================================")
print("OVERVIEW")
print("===================================\n")

print(overview_df)

print("\nExcel saved to:")
print(excel_output)

In [ ]:
# GOOD
# ============================================================
# ANALYSIS:
# DO NEWLY BROKEN RESOLUTION CONTEXT REQUIREMENTS
# CORRESPOND TO PREVIOUS OVERLAPS?
# ============================================================
#
# This script checks whether newly violated
# resolution context requirements are associated
# with structural overlaps between:
#
# - originally violated requirements
# - previously compliant requirements
#
# Overlap dimensions:
# - activities
# - resources
# - data
# - temporal constraints
#
# Output:
# - detailed overlap-impact analysis
# - Excel summary
#
# ============================================================

import json
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# Project root directory
BASE_DIR = Path(__file__).resolve().parent.parent
# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------

COMPLIANCE_REQUIREMENTS_DIR = (
    BASE_DIR
    / "data/input/compliance_requirements"
)

RESOLUTION_CONTEXT_DIR = (
    BASE_DIR
    / "data/input/resolution_context"
)

VIOLATIONS_AFTER_CHANGES_DIR = (
    BASE_DIR
    / "data/output/compliance_violations_after_changes"
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = (
    BASE_DIR
    / "data/output/overlap_resolution_context_analysis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# REGEX
# ============================================================

PATTERN_REGEX = re.compile(
    r'(\w+)\((.*?)\)'
)

# ============================================================
# HELPERS
# ============================================================

def split_arguments(arg_string):

    args = re.findall(
        r"'([^']*)'|([^,]+)",
        arg_string
    )

    cleaned = []

    for a, b in args:

        val = a if a else b

        cleaned.append(
            val.strip()
        )

    return cleaned

# ============================================================
# EXTRACT STRUCTURAL ELEMENTS
# ============================================================

def extract_elements(requirement_text):

    result = {
        "activities": set(),
        "resources": set(),
        "data": set(),
        "temporal": set(),
    }

    matches = PATTERN_REGEX.findall(
        requirement_text
    )

    for pattern_name, arg_string in matches:

        args = split_arguments(
            arg_string
        )

        # ----------------------------------------------------
        # Activity-only
        # ----------------------------------------------------

        if pattern_name in {
            "exists",
            "absence",
            "loop"
        }:

            if len(args) >= 1:

                result["activities"].add(
                    args[0]
                )

        # ----------------------------------------------------
        # Temporal
        # ----------------------------------------------------

        elif pattern_name in {
            "directly_follows",
            "leads_to",
            "precedence",
            "leads_to_absence",
            "precedence_absence",
            "parallel",
            "condition_directly_follows",
            "condition_eventually_follows"
        }:

            if len(args) >= 2:

                result["activities"].update(
                    [args[0], args[1]]
                )

                result["temporal"].add(
                    pattern_name
                )

        # ----------------------------------------------------
        # Resource
        # ----------------------------------------------------

        elif pattern_name == "executed_by":

            if len(args) >= 2:

                result["activities"].add(
                    args[0]
                )

                result["resources"].add(
                    args[1]
                )

        # ----------------------------------------------------
        # Timed
        # ----------------------------------------------------

        elif pattern_name == (
            "timed_alternative"
        ):

            if len(args) >= 3:

                result["activities"].update(
                    [args[0], args[1]]
                )

                result["temporal"].add(
                    "timed_alternative"
                )

        # ----------------------------------------------------
        # Data
        # ----------------------------------------------------

        elif pattern_name in {
            "activity_sends",
            "activity_receives",
            "data_leads_to_absence"
        }:

            if len(args) >= 2:

                result["activities"].add(
                    args[0]
                )

                result["data"].add(
                    args[1]
                )

    return result

# ============================================================
# LOAD JSON
# ============================================================

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

# ============================================================
# RESULTS
# ============================================================

rows = []

# ============================================================
# ITERATE SCENARIOS
# ============================================================

scenario_dirs = sorted(
    VIOLATIONS_AFTER_CHANGES_DIR.iterdir()
)

for scenario_dir in scenario_dirs:

    if not scenario_dir.is_dir():
        continue

    scenario_name = scenario_dir.name

    print("\n===================================")
    print("SCENARIO:", scenario_name)
    print("===================================\n")

    # --------------------------------------------------------
    # Load original violated requirements
    # --------------------------------------------------------

    violated_req_file = (
        COMPLIANCE_REQUIREMENTS_DIR
        / f"{scenario_name}.json"
    )

    if not violated_req_file.exists():

        print(
            "Missing compliance requirements."
        )

        continue

    violated_requirements = load_json(
        violated_req_file
    )

    # --------------------------------------------------------
    # Load resolution context
    # --------------------------------------------------------

    resolution_context_file = (
        RESOLUTION_CONTEXT_DIR
        / (
            f"{scenario_name}"
            f"_req_resolution_context.json"
        )
    )

    if not resolution_context_file.exists():

        print(
            "No resolution context."
        )

        continue

    resolution_context = load_json(
        resolution_context_file
    )

    # --------------------------------------------------------
    # PST violation reports
    # --------------------------------------------------------

    violation_reports = sorted(
        scenario_dir.glob("*.json")
    )

    for report_file in violation_reports:

        print(
            "Processing:",
            report_file.name
        )

        # ----------------------------------------------------
        # Extract original violated requirement
        # ----------------------------------------------------

        stem = report_file.stem

        parts = stem.split("_")

        original_requirement = None

        for part in parts:

            if (
                part.startswith("R")
                and len(part) > 1
                and part[1:].isdigit()
            ):

                original_requirement = part

                break

        if original_requirement is None:
            continue

        # ----------------------------------------------------
        # Load violation report
        # ----------------------------------------------------

        report_content = load_json(
            report_file
        )

        if isinstance(report_content, dict):

            violations_after = (
                report_content.get(
                    "violation_report",
                    []
                )
            )

        else:

            violations_after = report_content

        # ----------------------------------------------------
        # Collect newly violated resolution context reqs
        # ----------------------------------------------------

        broken_context_requirements = []

        for violation in violations_after:

            if not isinstance(
                violation,
                dict
            ):
                continue

            req_id = violation.get(
                "requirement_id"
            )

            if (
                req_id
                in resolution_context
            ):

                broken_context_requirements.append(
                    req_id
                )

        broken_context_requirements = sorted(
            set(
                broken_context_requirements
            )
        )

        # ----------------------------------------------------
        # Extract original violated structure
        # ----------------------------------------------------

        if (
            original_requirement
            not in violated_requirements
        ):

            continue

        original_elements = (
            extract_elements(
                violated_requirements[
                    original_requirement
                ]
            )
        )

        # ----------------------------------------------------
        # Compare with broken context reqs
        # ----------------------------------------------------

        for broken_req in (
            broken_context_requirements
        ):

            broken_elements = (
                extract_elements(
                    resolution_context[
                        broken_req
                    ]
                )
            )

            # ------------------------------------------------
            # Overlaps
            # ------------------------------------------------

            activity_overlap = sorted(
                original_elements[
                    "activities"
                ]
                &
                broken_elements[
                    "activities"
                ]
            )

            resource_overlap = sorted(
                original_elements[
                    "resources"
                ]
                &
                broken_elements[
                    "resources"
                ]
            )

            data_overlap = sorted(
                original_elements[
                    "data"
                ]
                &
                broken_elements[
                    "data"
                ]
            )

            temporal_overlap = sorted(
                original_elements[
                    "temporal"
                ]
                &
                broken_elements[
                    "temporal"
                ]
            )

            # ------------------------------------------------
            # Any overlap?
            # ------------------------------------------------

            any_overlap = any([
                activity_overlap,
                resource_overlap,
                data_overlap,
                temporal_overlap
            ])

            # ------------------------------------------------
            # Add row
            # ------------------------------------------------

            rows.append({

                "scenario":
                    scenario_name,

                "pst_file":
                    report_file.name,

                "original_violated_requirement":
                    original_requirement,

                "newly_broken_requirement":
                    broken_req,

                "has_overlap":
                    any_overlap,

                "activity_overlap":
                    ", ".join(
                        activity_overlap
                    ),

                "resource_overlap":
                    ", ".join(
                        resource_overlap
                    ),

                "data_overlap":
                    ", ".join(
                        data_overlap
                    ),

                "temporal_overlap":
                    ", ".join(
                        temporal_overlap
                    ),

                "num_activity_overlap":
                    len(
                        activity_overlap
                    ),

                "num_resource_overlap":
                    len(
                        resource_overlap
                    ),

                "num_data_overlap":
                    len(
                        data_overlap
                    ),

                "num_temporal_overlap":
                    len(
                        temporal_overlap
                    )
            })

# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

# ============================================================
# OVERVIEW STATISTICS
# ============================================================

if len(df) > 0:

    total_cases = len(df)

    overlap_cases = len(
        df[
            df["has_overlap"] == True
        ]
    )

    no_overlap_cases = (
        total_cases - overlap_cases
    )

    overview_df = pd.DataFrame([{

        "total_broken_context_cases":
            total_cases,

        "cases_with_overlap":
            overlap_cases,

        "cases_without_overlap":
            no_overlap_cases,

        "percentage_with_overlap":
            round(
                (
                    overlap_cases
                    / total_cases
                ) * 100,
                2
            )
    }])

else:

    overview_df = pd.DataFrame()

# ============================================================
# SAVE EXCEL
# ============================================================

excel_output = (
    OUTPUT_DIR
    / (
        "overlap_vs_broken_context"
        ".xlsx"
    )
)

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="detailed_results",
        index=False
    )

    overview_df.to_excel(
        writer,
        sheet_name="overview",
        index=False
    )

# ============================================================
# SUMMARY
# ============================================================

print("\n===================================")
print("SUMMARY")
print("===================================\n")

print(df)

print("\n===================================")
print("OVERVIEW")
print("===================================\n")

print(overview_df)

print("\nExcel saved to:")
print(excel_output)

In [ ]:
# ============================================================
# ANALYSIS STEP:
# CALCULATE GLOBAL COMPLIANCE VIOLATION STATISTICS
# ============================================================
#
# This script analyzes multiple compliance violation reports
# generated from process verification runs and computes
# aggregated statistics about violated compliance patterns.
#
# For each violation report, the script:
# - extracts violated compliance requirements
# - identifies the compliance patterns involved
# - computes global and per-file statistics
# - aggregates requirement-level information
#
# The script produces:
# - global violation statistics
# - per-scenario/file pattern distributions
# - requirement-level summaries
# - average/minimum/maximum violations
#
# Output:
# - aggregated violation statistics (.json)
#
# ============================================================

import json
import re
from pathlib import Path
from collections import Counter, defaultdict

# ============================================================
# DYNAMIC PROJECT ROOT DETECTION
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

BASE_DIR = CURRENT_DIR

while BASE_DIR.name != (
    "PTResolver"
):
    BASE_DIR = BASE_DIR.parent

print("Project root:")
print(BASE_DIR)

# ============================================================
# CONFIGURATION
# ============================================================

# Folder containing multiple violation JSON files

INPUT_FOLDER = (
    BASE_DIR
    / "data"
    / "output"
    / "compliance_violations_before_changes"
)

# Output summary file

OUTPUT_JSON = (
    BASE_DIR
    / "data"
    / "output"
    / "global_violation_statistics.json"
)

# Allowed compliance patterns

ALLOWED_PATTERNS = [
    "exists",
    "absence",
    "loop",
    "directly_follows",
    "leads_to",
    "precedence",
    "leads_to_absence",
    "precedence_absence",
    "parallel",
    "executed_by",
    "timed_alternative",
    "activity_sends",
    "activity_receives",
    "condition_directly_follows",
    "condition_eventually_follows",
    "data_leads_to_absence"
]

# ============================================================
# INITIALIZATION
# ============================================================

pattern_regex = re.compile(
    r'\b([a-zA-Z_][a-zA-Z0-9_]*)\s*\('
)

global_pattern_counter = Counter()

file_statistics = {}

requirement_statistics = defaultdict(list)

total_files = 0
total_violations = 0

# ============================================================
# ITERATE OVER ALL JSON FILES
# ============================================================

json_files = list(
    Path(INPUT_FOLDER).glob("*.json")
)

for json_file in json_files:

    total_files += 1

    print(f"Processing: {json_file.name}")

    try:

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            violations = json.load(f)

    except Exception as e:

        print(
            f"ERROR reading "
            f"{json_file.name}: {e}"
        )

        continue

    # --------------------------------------------------------
    # Skip invalid structures
    # --------------------------------------------------------

    if not isinstance(violations, list):

        print(
            f"Skipping {json_file.name}: "
            f"expected a list"
        )

        continue

    file_pattern_counter = Counter()

    file_violation_count = len(violations)

    total_violations += file_violation_count

    # --------------------------------------------------------
    # PROCESS EACH VIOLATION
    # --------------------------------------------------------

    for violation in violations:

        requirement_id = violation.get(
            "requirement_id",
            "UNKNOWN"
        )

        requirement = violation.get(
            "requirement",
            ""
        )

        found_patterns = pattern_regex.findall(
            requirement
        )

        valid_patterns = [
            p for p in found_patterns
            if p in ALLOWED_PATTERNS
        ]

        # ----------------------------------------------------
        # Global statistics
        # ----------------------------------------------------

        for pattern in valid_patterns:

            global_pattern_counter[
                pattern
            ] += 1

            file_pattern_counter[
                pattern
            ] += 1

        # ----------------------------------------------------
        # Store requirement-level info
        # ----------------------------------------------------

        requirement_statistics[
            json_file.name
        ].append({

            "requirement_id":
                requirement_id,

            "patterns":
                valid_patterns
        })

    # --------------------------------------------------------
    # STORE FILE-LEVEL STATS
    # --------------------------------------------------------

    file_statistics[json_file.name] = {

        "violations":
            file_violation_count,

        "pattern_distribution":
            dict(file_pattern_counter)
    }

# ============================================================
# ADDITIONAL VIOLATION STATISTICS
# ============================================================

violations_per_file = [
    stats["violations"]
    for stats in file_statistics.values()
]

if violations_per_file:

    average_violations = (
        sum(violations_per_file)
        / len(violations_per_file)
    )

    minimum_violations = min(
        violations_per_file
    )

    maximum_violations = max(
        violations_per_file
    )

else:

    average_violations = 0
    minimum_violations = 0
    maximum_violations = 0

# ============================================================
# PRINT GLOBAL RESULTS
# ============================================================

print("\n" + "=" * 70)

print(
    "GLOBAL COMPLIANCE VIOLATION STATISTICS"
)

print("=" * 70)

print(
    f"\nTotal JSON files processed: "
    f"{total_files}"
)

print(
    f"Total violations found:     "
    f"{total_violations}"
)

print(
    f"Average violations per scenario: "
    f"{average_violations:.2f}"
)

print(
    f"Minimum violations in a scenario: "
    f"{minimum_violations}"
)

print(
    f"Maximum violations in a scenario: "
    f"{maximum_violations}"
)

print("\nGLOBAL PATTERN DISTRIBUTION")

print("-" * 70)

for pattern in sorted(ALLOWED_PATTERNS):

    count = global_pattern_counter.get(
        pattern,
        0
    )

    percentage = (
        (count / total_violations) * 100
        if total_violations > 0 else 0
    )

    print(
        f"{pattern:<35} "
        f"{count:>6} "
        f"({percentage:.2f}%)"
    )

# ============================================================
# PRINT FILE-LEVEL SUMMARY
# ============================================================

print("\nFILE-LEVEL STATISTICS")

print("-" * 70)

for file_name, stats in file_statistics.items():

    print(f"\n{file_name}")

    print(
        f"  Violations: "
        f"{stats['violations']}"
    )

    if stats["pattern_distribution"]:

        for pattern, count in sorted(
            stats[
                "pattern_distribution"
            ].items()
        ):

            print(
                f"    {pattern:<30} "
                f"{count}"
            )

    else:

        print(
            "    No recognized patterns found"
        )

# ============================================================
# SAVE RESULTS
# ============================================================

summary = {

    "total_files":
        total_files,

    "total_violations":
        total_violations,

    "average_violations_per_scenario":
        average_violations,

    "minimum_violations_per_scenario":
        minimum_violations,

    "maximum_violations_per_scenario":
        maximum_violations,

    "global_pattern_distribution":
        dict(global_pattern_counter),

    "file_statistics":
        file_statistics,

    "requirement_statistics":
        dict(requirement_statistics)
}

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\n" + "=" * 70)

print(
    f"Statistics saved to:\n"
    f"{OUTPUT_JSON}"
)

print("=" * 70)